# Khởi tạo môi trường + đọc dữ liệu

In [ ]:
# ============================================================
# CELL 1: CÀI THƯ VIỆN
# ============================================================

!pip install -q transformers datasets accelerate evaluate scikit-learn pandas openpyxl tqdm py_vncorenlp

In [ ]:
# ============================================================
# CELL 2: MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# CELL 3: IMPORT + KHAI BÁO PATH
# ============================================================

import os
import re
import json
import random
import numpy as np
import pandas as pd

from collections import Counter, defaultdict
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Đổi path này nếu file của bạn nằm ở vị trí khác
DATA_PATH = "/content/drive/MyDrive/DACS/cleaned_data.xlsx"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Không tìm thấy file: {DATA_PATH}")

print("✅ Tìm thấy file dữ liệu:")
print(DATA_PATH)

In [ ]:
# ============================================================
# CELL 4: ĐỌC DỮ LIỆU
# ============================================================

df = pd.read_excel(DATA_PATH)

print("✅ Load dữ liệu thành công")
print("Shape:", df.shape)
print("\nCác cột:")
print(df.columns.tolist())

display(df.head(10))

In [ ]:
# ============================================================
# CELL 5: KIỂM TRA CỘT BẮT BUỘC
# ============================================================

REQUIRED_COLUMNS = {
    "source_index",
    "review_index",
    "unit_id",
    "unit_text_step2",
    "FINAL_REVIEWED_SENTIMENTS"
}

missing_cols = REQUIRED_COLUMNS - set(df.columns)

if missing_cols:
    raise ValueError(f"❌ Thiếu cột bắt buộc: {missing_cols}")

print("✅ Đủ toàn bộ cột bắt buộc")

In [ ]:
# ============================================================
# CELL 6: KIỂM TRA NULL / RỖNG
# ============================================================

check_cols = [
    "source_index",
    "review_index",
    "unit_id",
    "unit_text_step2",
    "FINAL_REVIEWED_SENTIMENTS"
]

null_report = df[check_cols].isna().sum()

print("Số giá trị null theo cột:")
display(null_report.to_frame("null_count"))

# Chuẩn hóa kiểu text để dùng an toàn ở các bước sau
df["unit_text_step2"] = df["unit_text_step2"].fillna("").astype(str).str.strip()
df["FINAL_REVIEWED_SENTIMENTS"] = df["FINAL_REVIEWED_SENTIMENTS"].fillna("").astype(str).str.strip()

empty_text_count = (df["unit_text_step2"] == "").sum()
empty_label_count = (df["FINAL_REVIEWED_SENTIMENTS"] == "").sum()

print(f"Số dòng text rỗng: {empty_text_count}")
print(f"Số dòng label rỗng: {empty_label_count}")

In [ ]:
# @title
# ============================================================
# CELL 6.5: LÀM SẠCH CƠ BẢN + KIỂM TRA VÀ XÓA TRÙNG LẶP
# ============================================================

print("===== BẮT ĐẦU LÀM SẠCH VÀ LỌC TRÙNG =====")

rows_before_clean = len(df)

# ------------------------------------------------------------
# 1. Chuẩn hóa nhẹ text và label
# ------------------------------------------------------------
df["unit_text_step2"] = (
    df["unit_text_step2"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

df["FINAL_REVIEWED_SENTIMENTS"] = (
    df["FINAL_REVIEWED_SENTIMENTS"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

# ------------------------------------------------------------
# 2. Loại bỏ dòng text rỗng hoặc label rỗng
# ------------------------------------------------------------
empty_text_mask = df["unit_text_step2"].eq("")
empty_label_mask = df["FINAL_REVIEWED_SENTIMENTS"].eq("")

num_empty_text = int(empty_text_mask.sum())
num_empty_label = int(empty_label_mask.sum())
num_empty_any = int((empty_text_mask | empty_label_mask).sum())

print("Số dòng text rỗng:", num_empty_text)
print("Số dòng label rỗng:", num_empty_label)
print("Số dòng bị loại vì text hoặc label rỗng:", num_empty_any)

df = df[
    ~(empty_text_mask | empty_label_mask)
].copy()

# ------------------------------------------------------------
# 3. Tạo key chuẩn hóa để kiểm tra duplicate
# ------------------------------------------------------------
def normalize_text_for_dedup(text: str) -> str:
    """
    Chuẩn hóa text chỉ để kiểm tra trùng:
    - lowercase
    - co khoảng trắng
    """
    if not isinstance(text, str):
        return ""

    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text


def normalize_label_for_dedup(label_text: str) -> str:
    """
    Chuẩn hóa chuỗi nhãn để kiểm tra trùng.
    Ví dụ:
    'ROOMS:NEGATIVE; HOTEL:POSITIVE'
    và
    'HOTEL:POSITIVE; ROOMS:NEGATIVE'
    sẽ được xem là cùng một bộ nhãn.
    """
    if not isinstance(label_text, str):
        return ""

    chunks = [
        chunk.strip().upper()
        for chunk in label_text.split(";")
        if chunk.strip()
    ]

    chunks = sorted(chunks)
    return "; ".join(chunks)


df["_dedup_text_key"] = df["unit_text_step2"].apply(
    normalize_text_for_dedup
)

df["_dedup_label_key"] = df["FINAL_REVIEWED_SENTIMENTS"].apply(
    normalize_label_for_dedup
)

# ------------------------------------------------------------
# 4. Kiểm tra các text giống nhau nhưng nhãn khác nhau
#    Không xóa tự động vì đây có thể là lỗi gán nhãn cần xem lại
# ------------------------------------------------------------
text_label_nunique = (
    df.groupby("_dedup_text_key")["_dedup_label_key"]
    .nunique()
)

conflicting_text_keys = text_label_nunique[
    text_label_nunique > 1
].index

conflicting_duplicates_df = df[
    df["_dedup_text_key"].isin(conflicting_text_keys)
].copy()

print("\nSố cụm nội dung giống nhau nhưng có nhãn khác nhau:",
      len(conflicting_text_keys))

print("Số dòng thuộc nhóm cần xem lại:",
      len(conflicting_duplicates_df))

if len(conflicting_duplicates_df) > 0:
    print("\n⚠️ Một số ví dụ text giống nhau nhưng nhãn khác nhau:")
    display(
        conflicting_duplicates_df[
            [
                "review_index",
                "unit_id",
                "unit_text_step2",
                "FINAL_REVIEWED_SENTIMENTS"
            ]
        ]
        .sort_values("unit_text_step2")
        .head(30)
    )
else:
    print("✅ Không phát hiện text trùng nhưng nhãn khác nhau.")

# ------------------------------------------------------------
# 5. Xóa duplicate hoàn toàn: cùng text chuẩn hóa + cùng nhãn chuẩn hóa
# ------------------------------------------------------------
duplicate_exact_mask = df.duplicated(
    subset=["_dedup_text_key", "_dedup_label_key"],
    keep="first"
)

num_exact_duplicates = int(duplicate_exact_mask.sum())

print("\nSố dòng trùng hoàn toàn về nội dung + nhãn:",
      num_exact_duplicates)

if num_exact_duplicates > 0:
    print("\nMột số dòng trùng hoàn toàn sẽ bị loại:")
    display(
        df.loc[
            duplicate_exact_mask,
            [
                "review_index",
                "unit_id",
                "unit_text_step2",
                "FINAL_REVIEWED_SENTIMENTS"
            ]
        ].head(30)
    )

df = df[
    ~duplicate_exact_mask
].copy()

# ------------------------------------------------------------
# 6. Xóa cột phụ và reset index
# ------------------------------------------------------------
df = df.drop(
    columns=[
        "_dedup_text_key",
        "_dedup_label_key"
    ]
).reset_index(drop=True)

rows_after_clean = len(df)
rows_removed_total = rows_before_clean - rows_after_clean

# ------------------------------------------------------------
# 7. Báo cáo tổng kết
# ------------------------------------------------------------
print("\n===== TỔNG KẾT SAU LÀM SẠCH =====")
print("Số dòng ban đầu:", rows_before_clean)
print("Số dòng sau làm sạch:", rows_after_clean)
print("Tổng số dòng đã loại:", rows_removed_total)

print("\n✅ Hoàn tất làm sạch cơ bản và lọc trùng lặp.")

In [ ]:
# ============================================================
# CELL 7: XEM RANDOM MỘT SỐ MẪU
# ============================================================

sample_df = df[
    ["review_index", "unit_id", "unit_text_step2", "FINAL_REVIEWED_SENTIMENTS"]
].sample(10, random_state=SEED)

display(sample_df)

# Kiểm tra nhãn và chuẩn hóa nhãn

In [ ]:
# ============================================================
# CELL 8: KHAI BÁO ASPECT + LABEL SCHEMA
# ============================================================

ASPECTS = [
    "HOTEL",
    "LOCATION",
    "ROOMS",
    "FACILITIES",
    "FOOD&DRINKS",
    "SERVICE"
]

LABELS = [
    "NONE",
    "POSITIVE",
    "NEGATIVE",
    "NEUTRAL",
    "CONFLICT"
]

LABEL2ID = {
    "NONE": 0,
    "POSITIVE": 1,
    "NEGATIVE": 2,
    "NEUTRAL": 3,
    "CONFLICT": 4
}

ID2LABEL = {v: k for k, v in LABEL2ID.items()}

print("ASPECTS:", ASPECTS)
print("LABEL2ID:", LABEL2ID)

In [ ]:
# ============================================================
# CELL 9: HÀM PARSE NHÃN
# ============================================================

def parse_label_string(label_text: str) -> dict:
    """
    Chuyển chuỗi nhãn:
        'HOTEL:POSITIVE; ROOMS:CONFLICT'
    thành dict đầy đủ 6 aspect:
        {
            'HOTEL': 'POSITIVE',
            'LOCATION': 'NONE',
            ...
        }
    """

    # Khởi tạo mặc định: mọi aspect đều NONE
    result = {aspect: "NONE" for aspect in ASPECTS}

    if not isinstance(label_text, str) or not label_text.strip():
        return result

    # Tách từng cặp aspect:sentiment
    chunks = [x.strip() for x in label_text.split(";") if x.strip()]

    for chunk in chunks:
        if ":" not in chunk:
            raise ValueError(f"Thiếu dấu ':' trong nhãn: {chunk}")

        aspect, sentiment = chunk.split(":", 1)

        aspect = aspect.strip().upper()
        sentiment = sentiment.strip().upper()

        if aspect not in ASPECTS:
            raise ValueError(f"Aspect không hợp lệ: {aspect}")

        if sentiment not in LABELS[1:]:
            raise ValueError(f"Sentiment không hợp lệ: {sentiment}")

        result[aspect] = sentiment

    return result

In [ ]:
# ============================================================
# CELL 10: TEST HÀM PARSE
# ============================================================

test_labels = [
    "HOTEL:POSITIVE",
    "ROOMS:CONFLICT; SERVICE:NEGATIVE",
    "LOCATION:POSITIVE; FOOD&DRINKS:NEUTRAL",
]

for x in test_labels:
    print("=" * 80)
    print("Input:", x)
    print("Parsed:", parse_label_string(x))

In [ ]:
# ============================================================
# CELL 11: KIỂM TRA NHÃN TOÀN BỘ DATASET
# ============================================================

label_errors = []

for idx, row in df.iterrows():
    label_text = row["FINAL_REVIEWED_SENTIMENTS"]

    try:
        _ = parse_label_string(label_text)
    except Exception as e:
        label_errors.append({
            "row_index": idx,
            "review_index": row["review_index"],
            "unit_id": row["unit_id"],
            "unit_text_step2": row["unit_text_step2"],
            "label_text": label_text,
            "error": str(e)
        })

print("Số dòng lỗi nhãn:", len(label_errors))

if label_errors:
    error_df = pd.DataFrame(label_errors)
    display(error_df.head(20))
else:
    print("✅ Không phát hiện lỗi format nhãn.")

In [ ]:
# ============================================================
# CELL 12: TẠO 6 CỘT LABEL DẠNG TEXT
# ============================================================

parsed_labels = df["FINAL_REVIEWED_SENTIMENTS"].apply(parse_label_string)

parsed_label_df = pd.DataFrame(parsed_labels.tolist())

# Đảm bảo đúng thứ tự aspect
parsed_label_df = parsed_label_df[ASPECTS]

# Ghép vào dataframe gốc
df_labeled = pd.concat(
    [df.reset_index(drop=True), parsed_label_df.reset_index(drop=True)],
    axis=1
)

print("✅ Đã tạo 6 cột nhãn aspect")
display(
    df_labeled[
        [
            "unit_text_step2",
            "FINAL_REVIEWED_SENTIMENTS",
            "HOTEL",
            "LOCATION",
            "ROOMS",
            "FACILITIES",
            "FOOD&DRINKS",
            "SERVICE",
        ]
    ].head(10)
)

In [ ]:
# ============================================================
# CELL 13: MÃ HÓA LABEL THÀNH SỐ
# ============================================================

for aspect in ASPECTS:
    df_labeled[f"{aspect}_id"] = df_labeled[aspect].map(LABEL2ID)

# Kiểm tra xem có bị NaN không
id_cols = [f"{aspect}_id" for aspect in ASPECTS]
missing_encoded = df_labeled[id_cols].isna().sum().sum()

print("Tổng số ô encode bị thiếu:", missing_encoded)

if missing_encoded == 0:
    print("✅ Encode label thành công.")
else:
    print("❌ Có lỗi khi encode label.")

display(
    df_labeled[
        [
            "unit_text_step2",
            "HOTEL_id",
            "LOCATION_id",
            "ROOMS_id",
            "FACILITIES_id",
            "FOOD&DRINKS_id",
            "SERVICE_id",
        ]
    ].head(10)
)

In [ ]:
# ============================================================
# CELL 14: ĐẾM SỐ ASPECT XUẤT HIỆN TRÊN MỖI UNIT
# ============================================================

df_labeled["num_active_aspects"] = (
    df_labeled[ASPECTS] != "NONE"
).sum(axis=1)

aspect_count_distribution = (
    df_labeled["num_active_aspects"]
    .value_counts()
    .sort_index()
)

print("Phân bố số aspect được gán trên mỗi unit:")
display(aspect_count_distribution.to_frame("num_units"))

In [ ]:
# ============================================================
# CELL 15: PHÂN BỐ SENTIMENT TỔNG THỂ
# ============================================================

sentiment_counter = Counter()

for aspect in ASPECTS:
    sentiment_counter.update(
        df_labeled.loc[df_labeled[aspect] != "NONE", aspect].tolist()
    )

sentiment_distribution_df = pd.DataFrame(
    sentiment_counter.items(),
    columns=["sentiment", "count"]
).sort_values("count", ascending=False)

display(sentiment_distribution_df)

In [ ]:
# ============================================================
# CELL 16: PHÂN BỐ SENTIMENT THEO TỪNG ASPECT
# ============================================================

aspect_sentiment_rows = []

for aspect in ASPECTS:
    counts = df_labeled[aspect].value_counts()

    row = {"aspect": aspect}
    for label in LABELS:
        row[label] = int(counts.get(label, 0))

    aspect_sentiment_rows.append(row)

aspect_sentiment_df = pd.DataFrame(aspect_sentiment_rows)

display(aspect_sentiment_df)

In [ ]:
# ============================================================
# CELL 17: RANDOM CHECK SAU PARSE
# ============================================================

cols_to_show = [
    "unit_text_step2",
    "FINAL_REVIEWED_SENTIMENTS",
    "HOTEL",
    "LOCATION",
    "ROOMS",
    "FACILITIES",
    "FOOD&DRINKS",
    "SERVICE"
]

display(
    df_labeled[cols_to_show]
    .sample(10, random_state=SEED)
)

In [ ]:
# ============================================================
# CELL 18: LƯU CHECKPOINT SAU TÁC VỤ 2
# ============================================================

OUTPUT_LABEL_READY_PATH = "/content/drive/MyDrive/DACS/absa_label_ready.csv"

df_labeled.to_csv(
    OUTPUT_LABEL_READY_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("✅ Đã lưu file:")
print(OUTPUT_LABEL_READY_PATH)

print("Shape cuối:", df_labeled.shape)

# Chuẩn bị dữ liệu cho training

In [ ]:
# ============================================================
# CELL 19: TÌM WORDSEGMENT THỦ CÔNG + BỎ "_" TRƯỚC KHI CHẠY VNCORE
# ============================================================

import re
import pandas as pd
from collections import Counter

# 1. Tìm toàn bộ token có dấu "_"
def extract_manual_segments(text: str):
    if not isinstance(text, str):
        return []
    return re.findall(r"\b[\wÀ-ỹ]+(?:_[\wÀ-ỹ]+)+\b", text.lower())

manual_segment_counter = Counter()

for text in df_labeled["unit_text_step2"].fillna("").astype(str):
    manual_segment_counter.update(extract_manual_segments(text))

# Danh sách wordsegment đã được "tự tay" thực hiện trong dữ liệu
MANUAL_SEGMENTS = sorted(
    manual_segment_counter.keys(),
    key=len,
    reverse=True
)

manual_segment_df = pd.DataFrame(
    manual_segment_counter.items(),
    columns=["manual_segment", "count"]
).sort_values("count", ascending=False).reset_index(drop=True)

print("Số wordsegment thủ công khác nhau:", len(MANUAL_SEGMENTS))
display(manual_segment_df.head(100))

# 2. Bỏ toàn bộ "_" trước khi đưa vào VnCore
df_labeled["text_before_vncore"] = (
    df_labeled["unit_text_step2"]
    .fillna("")
    .astype(str)
    .str.replace("_", " ", regex=False)
)

display(
    df_labeled[
        ["unit_text_step2", "text_before_vncore"]
    ].head(20)
)

In [ ]:
# ============================================================
# CELL 20: LOAD VNCORENLP TỪ GOOGLE DRIVE
# ============================================================

!pip install -q py_vncorenlp

import os
import py_vncorenlp

# Chỉnh lại nếu thư mục của bạn khác
VNCORE_DIR = "/content/drive/MyDrive/DACS/My_NLP_Models/vncorenlp"

if not os.path.exists(VNCORE_DIR):
    raise FileNotFoundError(f"Không tìm thấy thư mục VnCoreNLP: {VNCORE_DIR}")

vncore_segmenter = py_vncorenlp.VnCoreNLP(
    annotators=["wseg"],
    save_dir=VNCORE_DIR
)

print("✅ Load VnCoreNLP thành công")
print(VNCORE_DIR)

In [ ]:
# ============================================================
# CELL 21: CHẠY VNCORE WORD SEGMENTATION CHO TOÀN BỘ DỮ LIỆU
# ============================================================

from tqdm.auto import tqdm
tqdm.pandas()

def vncore_word_segment(text: str) -> str:
    if not isinstance(text, str):
        return ""

    text = text.strip()
    if not text:
        return ""

    try:
        output = vncore_segmenter.word_segment(text)

        if isinstance(output, list):
            return " ".join(
                sent.strip()
                for sent in output
                if isinstance(sent, str) and sent.strip()
            )

        if isinstance(output, str):
            return output.strip()

        return text

    except Exception as e:
        print(f"⚠️ Lỗi VnCore: {text[:100]} | {e}")
        return text

df_labeled["text_after_vncore"] = (
    df_labeled["text_before_vncore"]
    .progress_apply(vncore_word_segment)
)

print("✅ Hoàn tất chạy VnCoreNLP")

display(
    df_labeled[
        ["text_before_vncore", "text_after_vncore"]
    ].head(20)
)

In [ ]:
# ============================================================
# CELL 22: GÁN LẠI CÁC WORDSEGMENT THỦ CÔNG SAU VNCORE
# ============================================================

def reapply_manual_segments(text: str) -> str:
    if not isinstance(text, str):
        return ""

    result = text

    for segmented_token in MANUAL_SEGMENTS:
        plain_phrase = segmented_token.replace("_", " ")

        # Match cụm có khoảng trắng linh hoạt
        parts = [re.escape(part) for part in plain_phrase.split()]
        pattern = r"\b" + r"\s+".join(parts) + r"\b"

        result = re.sub(
            pattern,
            segmented_token,
            result,
            flags=re.IGNORECASE
        )

    result = re.sub(r"\s+", " ", result).strip()
    return result

df_labeled["text_model_input"] = (
    df_labeled["text_after_vncore"]
    .progress_apply(reapply_manual_segments)
)

print("✅ Đã gán lại các wordsegment thủ công")
display(
    df_labeled[
        [
            "unit_text_step2",
            "text_after_vncore",
            "text_model_input"
        ]
    ].head(30)
)

In [ ]:
# ============================================================
# CELL 23: LƯU CHECKPOINT SAU WORD SEGMENTATION
# ============================================================

SEGMENTED_READY_PATH = "/content/drive/MyDrive/DACS/absa_segmented_ready.csv"

df_labeled.to_csv(
    SEGMENTED_READY_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("✅ Đã lưu checkpoint:")
print(SEGMENTED_READY_PATH)

print("Shape:", df_labeled.shape)

display(
    df_labeled[
        [
            "unit_text_step2",
            "text_model_input",
            "FINAL_REVIEWED_SENTIMENTS"
        ]
    ].sample(10, random_state=42)
)

# Chia train/vaid/test một cách có cân bằng

In [ ]:
# ============================================================
# CELL 24: CÀI ITERATIVE STRATIFICATION + LOAD DATASET
# ============================================================

!pip install -q iterative-stratification

import os
import numpy as np
import pandas as pd

from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

SEGMENTED_READY_PATH = "/content/drive/MyDrive/DACS/absa_segmented_ready.csv"

if not os.path.exists(SEGMENTED_READY_PATH):
    raise FileNotFoundError(f"Không tìm thấy file: {SEGMENTED_READY_PATH}")

df_model = pd.read_csv(SEGMENTED_READY_PATH)

print("✅ Load dataset thành công")
print("Shape:", df_model.shape)
print("Các cột:")
print(df_model.columns.tolist())

display(df_model.head(10))

In [ ]:
# ============================================================
# CELL 25: KHAI BÁO SCHEMA + KIỂM TRA CỘT BẮT BUỘC
# ============================================================

ASPECTS = [
    "HOTEL",
    "LOCATION",
    "ROOMS",
    "FACILITIES",
    "FOOD&DRINKS",
    "SERVICE"
]

ID2LABEL = {
    0: "NONE",
    1: "POSITIVE",
    2: "NEGATIVE",
    3: "NEUTRAL",
    4: "CONFLICT"
}

LABEL_IDS_ACTIVE = [1, 2, 3, 4]

REQUIRED_COLUMNS = {
    "review_index",
    "unit_id",
    "text_model_input",
    "HOTEL_id",
    "LOCATION_id",
    "ROOMS_id",
    "FACILITIES_id",
    "FOOD&DRINKS_id",
    "SERVICE_id"
}

missing_cols = REQUIRED_COLUMNS - set(df_model.columns)

if missing_cols:
    raise ValueError(f"❌ Thiếu cột bắt buộc: {missing_cols}")

empty_input_count = (
    df_model["text_model_input"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

print("✅ Dataset đủ cột để split")
print("Số dòng text_model_input rỗng:", empty_input_count)
print("Số review khác nhau:", df_model["review_index"].nunique())

In [ ]:
# ============================================================
# CELL 26: TẠO LABEL PROFILE Ở MỨC REVIEW
# ============================================================

# Tạo các cột nhị phân aspect__sentiment ở mức unit
profile_cols = []

df_profile_source = df_model[["review_index"]].copy()

for aspect in ASPECTS:
    id_col = f"{aspect}_id"

    for label_id in LABEL_IDS_ACTIVE:
        label_name = ID2LABEL[label_id]
        col_name = f"{aspect}__{label_name}"

        df_profile_source[col_name] = (
            df_model[id_col].astype(int) == label_id
        ).astype(int)

        profile_cols.append(col_name)

# Gom từ unit-level lên review-level:
# Nếu review có ít nhất 1 unit chứa nhãn đó → 1
review_profile = (
    df_profile_source
    .groupby("review_index")[profile_cols]
    .max()
    .reset_index()
)

X_reviews = review_profile[["review_index"]].values
Y_reviews = review_profile[profile_cols].values

print("✅ Tạo review profile thành công")
print("Số review:", len(review_profile))
print("Số label profile:", len(profile_cols))

display(review_profile.head(10))

In [ ]:
# ============================================================
# CELL 27: THỐNG KÊ ĐỘ PHỦ ASPECT-SENTIMENT Ở MỨC REVIEW
# ============================================================

profile_support_df = pd.DataFrame({
    "profile_label": profile_cols,
    "num_reviews_containing_label": Y_reviews.sum(axis=0).astype(int)
}).sort_values(
    "num_reviews_containing_label",
    ascending=True
).reset_index(drop=True)

display(profile_support_df)

rare_profiles = profile_support_df[
    profile_support_df["num_reviews_containing_label"] < 10
]

print("Số profile xuất hiện dưới 10 review:", len(rare_profiles))
display(rare_profiles)

In [ ]:
# ============================================================
# CELL 28: SPLIT REVIEW TRAIN 80% / TEMP 20%
# ============================================================

SEED = 42

splitter_train_temp = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED
)

train_review_idx, temp_review_idx = next(
    splitter_train_temp.split(X_reviews, Y_reviews)
)

train_review_ids = review_profile.iloc[train_review_idx]["review_index"].tolist()
temp_review_profile = review_profile.iloc[temp_review_idx].reset_index(drop=True)

print("Số review train:", len(train_review_ids))
print("Số review temp:", len(temp_review_profile))

In [ ]:
# ============================================================
# CELL 29: SPLIT TEMP 50/50 → VALIDATION 10% / TEST 10%
# ============================================================

X_temp = temp_review_profile[["review_index"]].values
Y_temp = temp_review_profile[profile_cols].values

splitter_val_test = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=SEED
)

val_idx, test_idx = next(
    splitter_val_test.split(X_temp, Y_temp)
)

val_review_ids = temp_review_profile.iloc[val_idx]["review_index"].tolist()
test_review_ids = temp_review_profile.iloc[test_idx]["review_index"].tolist()

print("Số review validation:", len(val_review_ids))
print("Số review test:", len(test_review_ids))

print("\nTổng review sau chia:")
print(len(train_review_ids) + len(val_review_ids) + len(test_review_ids))
print("Review gốc:")
print(len(review_profile))

In [ ]:
# ============================================================
# CELL 30: TẠO DATAFRAME TRAIN / VALIDATION / TEST
# ============================================================

train_df = df_model[
    df_model["review_index"].isin(train_review_ids)
].copy()

val_df = df_model[
    df_model["review_index"].isin(val_review_ids)
].copy()

test_df = df_model[
    df_model["review_index"].isin(test_review_ids)
].copy()

print("Train rows:", len(train_df))
print("Validation rows:", len(val_df))
print("Test rows:", len(test_df))

print("\nTrain reviews:", train_df["review_index"].nunique())
print("Validation reviews:", val_df["review_index"].nunique())
print("Test reviews:", test_df["review_index"].nunique())

In [ ]:
# ============================================================
# CELL 31: KIỂM TRA REVIEW LEAKAGE
# ============================================================

train_review_set = set(train_df["review_index"].unique())
val_review_set = set(val_df["review_index"].unique())
test_review_set = set(test_df["review_index"].unique())

print("Train ∩ Validation:", len(train_review_set & val_review_set))
print("Train ∩ Test:", len(train_review_set & test_review_set))
print("Validation ∩ Test:", len(val_review_set & test_review_set))

if (
    len(train_review_set & val_review_set) == 0
    and len(train_review_set & test_review_set) == 0
    and len(val_review_set & test_review_set) == 0
):
    print("✅ Không có rò rỉ review_index giữa các tập.")
else:
    print("❌ Có rò rỉ review_index giữa các tập.")

In [ ]:
# ============================================================
# CELL 32: SO SÁNH PHÂN BỐ NHÃN GIỮA TRAIN / VAL / TEST
# ============================================================

def count_unit_level_aspect_sentiments(dataframe, split_name):
    rows = []

    for aspect in ASPECTS:
        id_col = f"{aspect}_id"

        for label_id in LABEL_IDS_ACTIVE:
            label_name = ID2LABEL[label_id]
            count = int((dataframe[id_col].astype(int) == label_id).sum())

            rows.append({
                "split": split_name,
                "aspect": aspect,
                "sentiment": label_name,
                "count": count
            })

    return pd.DataFrame(rows)

dist_train = count_unit_level_aspect_sentiments(train_df, "train")
dist_val = count_unit_level_aspect_sentiments(val_df, "validation")
dist_test = count_unit_level_aspect_sentiments(test_df, "test")

distribution_df = pd.concat(
    [dist_train, dist_val, dist_test],
    ignore_index=True
)

distribution_pivot = distribution_df.pivot_table(
    index=["aspect", "sentiment"],
    columns="split",
    values="count",
    fill_value=0
).reset_index()

display(distribution_pivot)

In [ ]:
# ============================================================
# CELL 33: SO SÁNH TỶ LỆ PHÂN BỐ NHÃN GIỮA CÁC TẬP
# ============================================================

distribution_ratio_df = distribution_df.copy()

split_sizes = {
    "train": len(train_df),
    "validation": len(val_df),
    "test": len(test_df)
}

distribution_ratio_df["ratio_per_unit"] = distribution_ratio_df.apply(
    lambda row: row["count"] / split_sizes[row["split"]],
    axis=1
)

distribution_ratio_pivot = distribution_ratio_df.pivot_table(
    index=["aspect", "sentiment"],
    columns="split",
    values="ratio_per_unit",
    fill_value=0
).reset_index()

display(distribution_ratio_pivot)

In [ ]:
# ============================================================
# CELL 34: CẢNH BÁO PROFILE BỊ MẤT Ở VAL / TEST
# ============================================================

missing_in_val = distribution_pivot[
    distribution_pivot.get("validation", 0) == 0
]

missing_in_test = distribution_pivot[
    distribution_pivot.get("test", 0) == 0
]

print("Nhãn không xuất hiện trong validation:")
display(missing_in_val)

print("Nhãn không xuất hiện trong test:")
display(missing_in_test)

In [ ]:
# ============================================================
# CELL 35: LƯU TRAIN / VALIDATION / TEST
# ============================================================

SPLIT_DIR = "/content/drive/MyDrive/DACS/absa_splits"
os.makedirs(SPLIT_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(SPLIT_DIR, "train.csv")
VAL_PATH = os.path.join(SPLIT_DIR, "validation.csv")
TEST_PATH = os.path.join(SPLIT_DIR, "test.csv")

train_df.to_csv(TRAIN_PATH, index=False, encoding="utf-8-sig")
val_df.to_csv(VAL_PATH, index=False, encoding="utf-8-sig")
test_df.to_csv(TEST_PATH, index=False, encoding="utf-8-sig")

# Lưu thêm review_profile và bảng phân bố để kiểm tra sau
PROFILE_PATH = os.path.join(SPLIT_DIR, "review_label_profile.csv")
DIST_PATH = os.path.join(SPLIT_DIR, "split_label_distribution.csv")

review_profile.to_csv(PROFILE_PATH, index=False, encoding="utf-8-sig")
distribution_pivot.to_csv(DIST_PATH, index=False, encoding="utf-8-sig")

print("✅ Đã lưu các file split:")
print(TRAIN_PATH)
print(VAL_PATH)
print(TEST_PATH)

print("\n✅ Đã lưu file hỗ trợ kiểm tra:")
print(PROFILE_PATH)
print(DIST_PATH)

# Tokenize bằng PhoBERT và tạo dataset train-ready

In [ ]:
# ============================================================
# CELL 36: LOAD TRAIN/VAL/TEST + TẠO VECTOR LABELS
# ============================================================

import os
import pandas as pd
import numpy as np

SPLIT_DIR = "/content/drive/MyDrive/DACS/absa_splits"

TRAIN_PATH = os.path.join(SPLIT_DIR, "train.csv")
VAL_PATH = os.path.join(SPLIT_DIR, "validation.csv")
TEST_PATH = os.path.join(SPLIT_DIR, "test.csv")

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

ASPECTS = [
    "HOTEL",
    "LOCATION",
    "ROOMS",
    "FACILITIES",
    "FOOD&DRINKS",
    "SERVICE"
]

LABEL_ID_COLUMNS = [f"{aspect}_id" for aspect in ASPECTS]

for split_name, dataframe in [
    ("train", train_df),
    ("validation", val_df),
    ("test", test_df)
]:
    required_cols = {"text_model_input"} | set(LABEL_ID_COLUMNS)
    missing_cols = required_cols - set(dataframe.columns)

    if missing_cols:
        raise ValueError(f"{split_name} thiếu cột: {missing_cols}")

    dataframe[LABEL_ID_COLUMNS] = dataframe[LABEL_ID_COLUMNS].astype(int)
    dataframe["labels"] = dataframe[LABEL_ID_COLUMNS].values.tolist()

print("✅ Load split data và tạo labels thành công")
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

display(
    train_df[
        ["text_model_input", "FINAL_REVIEWED_SENTIMENTS", "labels"]
    ].head(10)
)

In [ ]:
# ============================================================
# CELL 37: LOAD TOKENIZER + PHÂN TÍCH TOKEN LENGTH + CHỌN MAX_LENGTH
# ============================================================

!pip install -q transformers datasets

from transformers import AutoTokenizer
from tqdm.auto import tqdm

PHOBERT_MODEL_NAME = "vinai/phobert-base"

tokenizer = AutoTokenizer.from_pretrained(
    PHOBERT_MODEL_NAME,
    use_fast=False
)

all_texts = pd.concat([
    train_df["text_model_input"],
    val_df["text_model_input"],
    test_df["text_model_input"]
], ignore_index=True).fillna("").astype(str).tolist()

token_lengths = []

for text in tqdm(all_texts, desc="Đang đo độ dài token"):
    encoded = tokenizer(
        text,
        add_special_tokens=True,
        truncation=False
    )
    token_lengths.append(len(encoded["input_ids"]))

length_series = pd.Series(token_lengths)

stats = length_series.describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
)

display(stats.to_frame("token_length"))

p99 = int(length_series.quantile(0.99))

if p99 <= 64:
    MAX_LENGTH = 64
elif p99 <= 128:
    MAX_LENGTH = 128
else:
    MAX_LENGTH = 256

print("✅ Load tokenizer thành công")
print("Tokenizer:", tokenizer.__class__.__name__)
print("Vocab size:", tokenizer.vocab_size)

print("\nĐộ dài token:")
print("Số mẫu > 64 :", int((length_series > 64).sum()))
print("Số mẫu > 128:", int((length_series > 128).sum()))
print("Số mẫu > 256:", int((length_series > 256).sum()))

print("\nMAX_LENGTH được đề xuất:", MAX_LENGTH)

In [ ]:
# ============================================================
# CELL 38: TẠO DATASET + TOKENIZE TOÀN BỘ
# ============================================================

from datasets import Dataset, DatasetDict

KEEP_COLUMNS = [
    "text_model_input",
    "labels"
]

dataset_dict = DatasetDict({
    "train": Dataset.from_pandas(
        train_df[KEEP_COLUMNS].reset_index(drop=True)
    ),
    "validation": Dataset.from_pandas(
        val_df[KEEP_COLUMNS].reset_index(drop=True)
    ),
    "test": Dataset.from_pandas(
        test_df[KEEP_COLUMNS].reset_index(drop=True)
    )
})

def tokenize_batch(batch):
    return tokenizer(
        batch["text_model_input"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

tokenized_datasets = dataset_dict.map(
    tokenize_batch,
    batched=True,
    desc="Đang tokenize dữ liệu"
)

print("✅ Tokenize hoàn tất")
print(tokenized_datasets)

print("\nVí dụ sau tokenize:")
print(tokenized_datasets["train"][0])

In [ ]:
# ============================================================
# CELL 39: SET FORMAT PYTORCH + KIỂM TRA OUTPUT
# ============================================================

tokenized_datasets = tokenized_datasets.remove_columns([
    "text_model_input"
])

tokenized_datasets.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)

sample = tokenized_datasets["train"][0]

print("✅ Dataset đã sẵn sàng cho PyTorch")
print(tokenized_datasets)

print("\nKiểm tra 1 sample:")
print("input_ids shape:", sample["input_ids"].shape)
print("attention_mask shape:", sample["attention_mask"].shape)
print("labels:", sample["labels"])
print("labels shape:", sample["labels"].shape)

num_truncated_like = int((length_series > MAX_LENGTH).sum())
ratio_truncated_like = num_truncated_like / len(length_series)

print("\nKiểm tra truncate:")
print("MAX_LENGTH:", MAX_LENGTH)
print("Số mẫu dài hơn MAX_LENGTH:", num_truncated_like)
print("Tỷ lệ có thể bị truncate:", round(ratio_truncated_like * 100, 4), "%")

In [ ]:
# ============================================================
# CELL 40: LƯU TOKENIZED DATASET
# ============================================================

TOKENIZED_SAVE_DIR = "/content/drive/MyDrive/DACS/absa_tokenized_datasets"

tokenized_datasets.save_to_disk(TOKENIZED_SAVE_DIR)

print("✅ Đã lưu tokenized datasets:")
print(TOKENIZED_SAVE_DIR)

# PhoBERT ABSA + Optuna Hyperparameter Search + Loss Curve

In [ ]:
# ============================================================
# CELL 41: LOAD DATASET + CẤU HÌNH CHUNG + GPU
# ============================================================

!pip install -q optuna

import os
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from datasets import load_from_disk
from transformers import AutoModel

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    print("✅ GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ Đang chạy bằng CPU. HPO sẽ rất chậm.")

# ------------------------------------------------------------
# 1. Load tokenized dataset
# ------------------------------------------------------------
TOKENIZED_SAVE_DIR = "/content/drive/MyDrive/DACS/absa_tokenized_datasets"

if not os.path.exists(TOKENIZED_SAVE_DIR):
    raise FileNotFoundError(f"Không tìm thấy tokenized dataset: {TOKENIZED_SAVE_DIR}")

tokenized_datasets = load_from_disk(TOKENIZED_SAVE_DIR)

print(tokenized_datasets)

# ------------------------------------------------------------
# 2. Cấu hình label
# ------------------------------------------------------------
ASPECTS = [
    "HOTEL",
    "LOCATION",
    "ROOMS",
    "FACILITIES",
    "FOOD&DRINKS",
    "SERVICE"
]

ID2LABEL = {
    0: "NONE",
    1: "POSITIVE",
    2: "NEGATIVE",
    3: "NEUTRAL",
    4: "CONFLICT"
}

NUM_ASPECTS = len(ASPECTS)
NUM_CLASSES = len(ID2LABEL)

PHOBERT_MODEL_NAME = "vinai/phobert-base"
MAX_LENGTH = 64

print("\nNUM_ASPECTS:", NUM_ASPECTS)
print("NUM_CLASSES:", NUM_CLASSES)

In [ ]:
# ============================================================
# CELL 42: CLASS WEIGHTS + MODEL CLASS + METRICS
# ============================================================

from sklearn.metrics import f1_score, accuracy_score

# ------------------------------------------------------------
# 1. Lấy label train để tính class weight
# ------------------------------------------------------------
train_labels_np = np.array(tokenized_datasets["train"]["labels"])

def build_class_weights(max_class_weight: float = 5.0) -> torch.Tensor:
    """
    Tính class weights riêng cho từng aspect.
    Output shape: [NUM_ASPECTS, NUM_CLASSES]
    """
    all_weights = []

    for aspect_idx in range(NUM_ASPECTS):
        y = train_labels_np[:, aspect_idx]

        counts = np.bincount(y, minlength=NUM_CLASSES)
        total = counts.sum()

        weights = total / (NUM_CLASSES * np.maximum(counts, 1))
        weights = np.clip(weights, 0.0, max_class_weight)

        # Chuẩn hóa để trung bình weight xấp xỉ 1
        weights = weights / weights.mean()

        all_weights.append(weights)

    return torch.tensor(
        np.array(all_weights),
        dtype=torch.float32
    )

# ------------------------------------------------------------
# 2. Model
# ------------------------------------------------------------
class PhoBERTMultiAspectClassifier(nn.Module):
    def __init__(
        self,
        pretrained_model_name: str,
        num_aspects: int,
        num_classes: int,
        class_weights: torch.Tensor,
        dropout_prob: float = 0.2
    ):
        super().__init__()

        self.num_aspects = num_aspects
        self.num_classes = num_classes

        self.encoder = AutoModel.from_pretrained(pretrained_model_name)
        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(dropout_prob)
        self.classifier = nn.Linear(
            hidden_size,
            num_aspects * num_classes
        )

        self.register_buffer("class_weights", class_weights)

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        labels=None
    ):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)

        logits = self.classifier(cls_output)
        logits = logits.view(
            -1,
            self.num_aspects,
            self.num_classes
        )

        loss = None

        if labels is not None:
            labels = labels.long()
            aspect_losses = []

            for aspect_idx in range(self.num_aspects):
                loss_fn = nn.CrossEntropyLoss(
                    weight=self.class_weights[aspect_idx]
                )

                aspect_loss = loss_fn(
                    logits[:, aspect_idx, :],
                    labels[:, aspect_idx]
                )

                aspect_losses.append(aspect_loss)

            loss = torch.stack(aspect_losses).mean()

        return {
            "loss": loss,
            "logits": logits
        }

# ------------------------------------------------------------
# 3. Metrics
# ------------------------------------------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    metrics = {}

    aspect_macro_f1_scores = []
    aspect_accuracy_scores = []

    for aspect_idx, aspect in enumerate(ASPECTS):
        y_true = labels[:, aspect_idx]
        y_pred = preds[:, aspect_idx]

        macro_f1 = f1_score(
            y_true,
            y_pred,
            labels=list(range(NUM_CLASSES)),
            average="macro",
            zero_division=0
        )

        accuracy = accuracy_score(y_true, y_pred)

        aspect_macro_f1_scores.append(macro_f1)
        aspect_accuracy_scores.append(accuracy)

        metrics[f"{aspect.lower()}_macro_f1"] = float(macro_f1)
        metrics[f"{aspect.lower()}_accuracy"] = float(accuracy)

    metrics["mean_aspect_macro_f1"] = float(
        np.mean(aspect_macro_f1_scores)
    )

    metrics["mean_aspect_accuracy"] = float(
        np.mean(aspect_accuracy_scores)
    )

    metrics["exact_match"] = float(
        np.mean(np.all(preds == labels, axis=1))
    )

    return metrics

print("✅ Đã khai báo class weights, model class và metrics")

In [ ]:
# ============================================================
# CELL 43 MỚI: HPO ƯU TIÊN LOSS CURVE ỔN ĐỊNH HƠN
# ============================================================

from transformers import (
    TrainingArguments,
    Trainer,
    default_data_collator
)
import inspect

# Tách thư mục output để không ghi đè kết quả cũ
HPO_OUTPUT_DIR = "/content/drive/MyDrive/DACS/phobert_absa_hpo_loss_balanced"

# ------------------------------------------------------------
# 1. Model init cho từng trial
# ------------------------------------------------------------
def model_init(trial=None):
    """
    Mỗi trial khởi tạo 1 model mới.
    Lần HPO này ưu tiên các cấu hình regularization mạnh hơn.
    """

    if trial is None:
        dropout_prob = 0.3
        max_class_weight = 4.0
    else:
        dropout_prob = trial.suggest_categorical(
            "dropout_prob",
            [0.2, 0.3, 0.4]
        )

        max_class_weight = trial.suggest_categorical(
            "max_class_weight",
            [3.0, 4.0, 5.0]
        )

    trial_class_weights = build_class_weights(
        max_class_weight=max_class_weight
    )

    return PhoBERTMultiAspectClassifier(
        pretrained_model_name=PHOBERT_MODEL_NAME,
        num_aspects=NUM_ASPECTS,
        num_classes=NUM_CLASSES,
        class_weights=trial_class_weights,
        dropout_prob=dropout_prob
    )

# ------------------------------------------------------------
# 2. TrainingArguments nền cho HPO
# ------------------------------------------------------------
hpo_args_kwargs = dict(
    output_dir=HPO_OUTPUT_DIR,

    # Giá trị nền, Optuna sẽ override khi trial chạy
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,

    weight_decay=0.05,
    warmup_ratio=0.1,

    logging_strategy="epoch",
    save_strategy="no",

    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED
)

training_args_signature = inspect.signature(
    TrainingArguments.__init__
).parameters

if "eval_strategy" in training_args_signature:
    hpo_args_kwargs["eval_strategy"] = "epoch"
else:
    hpo_args_kwargs["evaluation_strategy"] = "epoch"

hpo_training_args = TrainingArguments(**hpo_args_kwargs)

# ------------------------------------------------------------
# 3. Trainer HPO
# ------------------------------------------------------------
hpo_trainer = Trainer(
    model_init=model_init,
    args=hpo_training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=default_data_collator,
    compute_metrics=compute_metrics
)

# ------------------------------------------------------------
# 4. Search space mới: học chậm hơn, regularization mạnh hơn
# ------------------------------------------------------------
def hp_space(trial):
    return {
        "learning_rate": trial.suggest_float(
            "learning_rate",
            5e-6,
            2e-5,
            log=True
        ),

        "per_device_train_batch_size": trial.suggest_categorical(
            "per_device_train_batch_size",
            [16, 32]
        ),

        "weight_decay": trial.suggest_categorical(
            "weight_decay",
            [0.01, 0.05, 0.1]
        ),

        "warmup_ratio": trial.suggest_categorical(
            "warmup_ratio",
            [0.05, 0.1]
        )
    }

# ------------------------------------------------------------
# 5. Objective mới: cân bằng F1 và Validation Loss
# ------------------------------------------------------------
ALPHA_LOSS = 0.25

def compute_objective(metrics):
    macro_f1 = metrics["eval_mean_aspect_macro_f1"]
    val_loss = metrics["eval_loss"]

    composite_score = macro_f1 - ALPHA_LOSS * val_loss
    return composite_score

# ------------------------------------------------------------
# 6. Chạy HPO
# ------------------------------------------------------------
N_TRIALS = 8

best_run = hpo_trainer.hyperparameter_search(
    direction="maximize",
    backend="optuna",
    hp_space=hp_space,
    compute_objective=compute_objective,
    n_trials=N_TRIALS
)

print("✅ HPO lần 2 hoàn tất")

print("\nBest composite objective:")
print(best_run.objective)

print("\nBest hyperparameters:")
print(best_run.hyperparameters)

In [ ]:
# ============================================================
# CELL 44 MỚI: TRAIN FINAL MODEL ƯU TIÊN VALIDATION LOSS
# ============================================================

from transformers import EarlyStoppingCallback

best_params = best_run.hyperparameters

BEST_LEARNING_RATE = best_params["learning_rate"]
BEST_BATCH_SIZE = best_params["per_device_train_batch_size"]
BEST_WEIGHT_DECAY = best_params["weight_decay"]
BEST_WARMUP_RATIO = best_params["warmup_ratio"]
BEST_DROPOUT = best_params["dropout_prob"]
BEST_MAX_CLASS_WEIGHT = best_params["max_class_weight"]

print("Best params used for final training:")
print(json.dumps(best_params, indent=2))

# ------------------------------------------------------------
# 1. Tạo class weights theo best params
# ------------------------------------------------------------
best_class_weights = build_class_weights(
    max_class_weight=BEST_MAX_CLASS_WEIGHT
)

class_weight_df = pd.DataFrame(
    best_class_weights.numpy(),
    index=ASPECTS,
    columns=[ID2LABEL[i] for i in range(NUM_CLASSES)]
)

print("\nClass weights final:")
display(class_weight_df)

# ------------------------------------------------------------
# 2. Khởi tạo final model mới từ đầu
# ------------------------------------------------------------
final_model = PhoBERTMultiAspectClassifier(
    pretrained_model_name=PHOBERT_MODEL_NAME,
    num_aspects=NUM_ASPECTS,
    num_classes=NUM_CLASSES,
    class_weights=best_class_weights,
    dropout_prob=BEST_DROPOUT
)

# ------------------------------------------------------------
# 3. TrainingArguments final
#    Khác run trước:
#    - load best model theo eval_loss
#    - early stopping cũng bám theo eval_loss
# ------------------------------------------------------------
FINAL_OUTPUT_DIR = "/content/drive/MyDrive/DACS/phobert_absa_final_loss_balanced"

final_args_kwargs = dict(
    output_dir=FINAL_OUTPUT_DIR,

    learning_rate=BEST_LEARNING_RATE,
    per_device_train_batch_size=BEST_BATCH_SIZE,
    per_device_eval_batch_size=32,
    num_train_epochs=10,

    weight_decay=BEST_WEIGHT_DECAY,
    warmup_ratio=BEST_WARMUP_RATIO,

    logging_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    save_total_limit=2,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED
)

if "eval_strategy" in training_args_signature:
    final_args_kwargs["eval_strategy"] = "epoch"
else:
    final_args_kwargs["evaluation_strategy"] = "epoch"

final_training_args = TrainingArguments(**final_args_kwargs)

# ------------------------------------------------------------
# 4. Final trainer
# ------------------------------------------------------------
final_trainer = Trainer(
    model=final_model,
    args=final_training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=default_data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ]
)

# ------------------------------------------------------------
# 5. Train
# ------------------------------------------------------------
final_train_result = final_trainer.train()

print("\n✅ Final training hoàn tất")

# ------------------------------------------------------------
# 6. Evaluate validation và test
# ------------------------------------------------------------
validation_metrics = final_trainer.evaluate(
    tokenized_datasets["validation"],
    metric_key_prefix="validation"
)

test_metrics = final_trainer.evaluate(
    tokenized_datasets["test"],
    metric_key_prefix="test"
)

print("\nValidation metrics:")
for k, v in validation_metrics.items():
    print(f"{k}: {v}")

print("\nTest metrics:")
for k, v in test_metrics.items():
    print(f"{k}: {v}")

In [ ]:
# ============================================================
# CELL 45: VẼ TRAIN LOSS / VALIDATION LOSS
# ============================================================

import matplotlib.pyplot as plt

log_history = final_trainer.state.log_history

train_epochs = []
train_losses = []

val_epochs = []
val_losses = []

val_f1_epochs = []
val_macro_f1s = []

for log in log_history:
    if "loss" in log and "epoch" in log:
        train_epochs.append(log["epoch"])
        train_losses.append(log["loss"])

    if "eval_loss" in log and "epoch" in log:
        val_epochs.append(log["epoch"])
        val_losses.append(log["eval_loss"])

    if "eval_mean_aspect_macro_f1" in log and "epoch" in log:
        val_f1_epochs.append(log["epoch"])
        val_macro_f1s.append(log["eval_mean_aspect_macro_f1"])

# ------------------------------------------------------------
# 1. Loss curve
# ------------------------------------------------------------
plt.figure(figsize=(8, 5))
plt.plot(train_epochs, train_losses, marker="o", label="Train Loss")
plt.plot(val_epochs, val_losses, marker="o", label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train Loss vs Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

# ------------------------------------------------------------
# 2. Validation Macro F1 curve
# ------------------------------------------------------------
plt.figure(figsize=(8, 5))
plt.plot(
    val_f1_epochs,
    val_macro_f1s,
    marker="o",
    label="Validation Mean Aspect Macro F1"
)
plt.xlabel("Epoch")
plt.ylabel("Macro F1")
plt.title("Validation Mean Aspect Macro F1")
plt.legend()
plt.grid(True)
plt.show()

# ------------------------------------------------------------
# 3. Bảng log gọn
# ------------------------------------------------------------
training_curve_df = pd.DataFrame({
    "epoch": val_epochs,
    "validation_loss": val_losses,
    "validation_mean_macro_f1": val_macro_f1s
})

display(training_curve_df)

In [ ]:
# ============================================================
# CELL 46 MỚI: LƯU MODEL LOSS-BALANCED + METRICS + LOG
# ============================================================

FINAL_SAVE_DIR = "/content/drive/MyDrive/DACS/phobert_absa_final_loss_balanced/final_artifacts"
os.makedirs(FINAL_SAVE_DIR, exist_ok=True)

# ------------------------------------------------------------
# 1. Lưu model state
# ------------------------------------------------------------
MODEL_PATH = os.path.join(
    FINAL_SAVE_DIR,
    "phobert_absa_multiaspect_loss_balanced.pt"
)

torch.save(
    {
        "model_state_dict": final_trainer.model.state_dict(),
        "phobert_model_name": PHOBERT_MODEL_NAME,
        "num_aspects": NUM_ASPECTS,
        "num_classes": NUM_CLASSES,
        "aspects": ASPECTS,
        "id2label": ID2LABEL,
        "max_length": MAX_LENGTH,
        "best_hyperparameters": best_params,
        "class_weights": best_class_weights.cpu().numpy().tolist(),
        "selection_policy": {
            "hpo_objective": "macro_f1 - 0.25 * eval_loss",
            "final_checkpoint_metric": "eval_loss"
        }
    },
    MODEL_PATH
)

# ------------------------------------------------------------
# 2. Lưu best hyperparameters
# ------------------------------------------------------------
BEST_PARAMS_PATH = os.path.join(
    FINAL_SAVE_DIR,
    "best_hyperparameters_loss_balanced.json"
)

with open(BEST_PARAMS_PATH, "w", encoding="utf-8") as f:
    json.dump(
        best_params,
        f,
        ensure_ascii=False,
        indent=2
    )

# ------------------------------------------------------------
# 3. Lưu final metrics
# ------------------------------------------------------------
METRICS_PATH = os.path.join(
    FINAL_SAVE_DIR,
    "final_metrics_loss_balanced.json"
)

with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(
        {
            "validation_metrics": validation_metrics,
            "test_metrics": test_metrics
        },
        f,
        ensure_ascii=False,
        indent=2
    )

# ------------------------------------------------------------
# 4. Lưu trainer log history
# ------------------------------------------------------------
LOG_HISTORY_PATH = os.path.join(
    FINAL_SAVE_DIR,
    "trainer_log_history_loss_balanced.json"
)

with open(LOG_HISTORY_PATH, "w", encoding="utf-8") as f:
    json.dump(
        final_trainer.state.log_history,
        f,
        ensure_ascii=False,
        indent=2
    )

print("✅ Đã lưu artifacts của run loss-balanced:")
print(MODEL_PATH)
print(BEST_PARAMS_PATH)
print(METRICS_PATH)
print(LOG_HISTORY_PATH)

# Đánh giá mô hình trên tập test

In [ ]:
# ============================================================
# CELL 47: PREDICT TRÊN TEST SET
# ============================================================

import numpy as np
import pandas as pd

# 1. Chạy dự đoán trên tập test
test_prediction_output = final_trainer.predict(
    tokenized_datasets["test"]
)

# 2. Lấy logits và labels thật
test_logits = test_prediction_output.predictions
test_labels = test_prediction_output.label_ids

# 3. Chuyển logits -> nhãn dự đoán
test_preds = np.argmax(test_logits, axis=-1)

print("✅ Predict test set hoàn tất")
print("test_logits shape:", test_logits.shape)
print("test_labels shape:", test_labels.shape)
print("test_preds shape:", test_preds.shape)

In [ ]:
# ============================================================
# CELL 48: TÍNH METRIC TỔNG QUÁT TRÊN TEST SET
# ============================================================

from sklearn.metrics import f1_score, accuracy_score

ASPECTS = [
    "HOTEL",
    "LOCATION",
    "ROOMS",
    "FACILITIES",
    "FOOD&DRINKS",
    "SERVICE"
]

ID2LABEL = {
    0: "NONE",
    1: "POSITIVE",
    2: "NEGATIVE",
    3: "NEUTRAL",
    4: "CONFLICT"
}

NUM_CLASSES = len(ID2LABEL)

aspect_summary_rows = []

for aspect_idx, aspect in enumerate(ASPECTS):
    y_true = test_labels[:, aspect_idx]
    y_pred = test_preds[:, aspect_idx]

    macro_f1 = f1_score(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        average="weighted",
        zero_division=0
    )

    accuracy = accuracy_score(y_true, y_pred)

    aspect_summary_rows.append({
        "aspect": aspect,
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1
    })

aspect_summary_df = pd.DataFrame(aspect_summary_rows)

mean_aspect_macro_f1 = aspect_summary_df["macro_f1"].mean()
mean_aspect_accuracy = aspect_summary_df["accuracy"].mean()
mean_aspect_weighted_f1 = aspect_summary_df["weighted_f1"].mean()

exact_match = np.mean(
    np.all(test_preds == test_labels, axis=1)
)

print("===== TEST SET OVERALL METRICS =====")
print(f"Mean Aspect Accuracy   : {mean_aspect_accuracy:.4f}")
print(f"Mean Aspect Macro F1   : {mean_aspect_macro_f1:.4f}")
print(f"Mean Aspect Weighted F1: {mean_aspect_weighted_f1:.4f}")
print(f"Exact Match            : {exact_match:.4f}")

print("\n===== TEST METRICS THEO TỪNG ASPECT =====")
display(
    aspect_summary_df.sort_values(
        "macro_f1",
        ascending=False
    ).reset_index(drop=True)
)

In [ ]:
# ============================================================
# CELL 49: CLASSIFICATION REPORT CHO TỪNG ASPECT
# ============================================================

from sklearn.metrics import classification_report

classification_reports = {}

for aspect_idx, aspect in enumerate(ASPECTS):
    y_true = test_labels[:, aspect_idx]
    y_pred = test_preds[:, aspect_idx]

    report_dict = classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=[ID2LABEL[i] for i in range(NUM_CLASSES)],
        output_dict=True,
        zero_division=0
    )

    report_df = pd.DataFrame(report_dict).transpose()

    classification_reports[aspect] = report_df

    print("=" * 100)
    print(f"CLASSIFICATION REPORT — {aspect}")
    display(report_df.round(4))

In [ ]:
# ============================================================
# CELL 50: CONFUSION MATRIX CHO TỪNG ASPECT
# ============================================================

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

LABEL_NAMES = [ID2LABEL[i] for i in range(NUM_CLASSES)]

for aspect_idx, aspect in enumerate(ASPECTS):
    y_true = test_labels[:, aspect_idx]
    y_pred = test_preds[:, aspect_idx]

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES))
    )

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=LABEL_NAMES
    )

    plt.figure(figsize=(7, 6))
    disp.plot(
        values_format="d",
        cmap=None,
        ax=plt.gca(),
        colorbar=False
    )
    plt.title(f"Confusion Matrix — {aspect}")
    plt.xticks(rotation=45)
    plt.grid(False)
    plt.show()

In [ ]:
# ============================================================
# CELL 51: TẠO BẢNG DỰ ĐOÁN CHI TIẾT TEST SET
# ============================================================

import os

TEST_CSV_PATH = "/content/drive/MyDrive/DACS/absa_splits/test.csv"

test_raw_df = pd.read_csv(TEST_CSV_PATH).reset_index(drop=True)

if len(test_raw_df) != len(test_preds):
    raise ValueError(
        f"Số dòng test.csv ({len(test_raw_df)}) "
        f"khác số prediction ({len(test_preds)})"
    )

def decode_label_row(label_row):
    """
    Chuyển 1 vector [6] thành chuỗi nhãn dễ đọc.
    Bỏ qua các aspect NONE.
    """
    pairs = []

    for aspect_idx, label_id in enumerate(label_row):
        label_name = ID2LABEL[int(label_id)]

        if label_name != "NONE":
            pairs.append(f"{ASPECTS[aspect_idx]}:{label_name}")

    if not pairs:
        return "NONE"

    return "; ".join(pairs)

test_result_df = test_raw_df[
    [
        "review_index",
        "unit_id",
        "unit_text_step2",
        "text_model_input",
        "FINAL_REVIEWED_SENTIMENTS"
    ]
].copy()

test_result_df["ground_truth_decoded"] = [
    decode_label_row(row)
    for row in test_labels
]

test_result_df["prediction_decoded"] = [
    decode_label_row(row)
    for row in test_preds
]

test_result_df["exact_match"] = np.all(
    test_preds == test_labels,
    axis=1
)

test_result_df["num_wrong_aspects"] = (
    test_preds != test_labels
).sum(axis=1)

print("✅ Đã tạo bảng kết quả dự đoán chi tiết")
print("Số dòng test:", len(test_result_df))
print("Exact match:", round(test_result_df["exact_match"].mean(), 4))

display(test_result_df.head(20))

In [ ]:
# ============================================================
# CELL 52: XEM DỰ ĐOÁN ĐÚNG / SAI TRÊN TEST SET
# ============================================================

print("===== MẪU DỰ ĐOÁN ĐÚNG HOÀN TOÀN =====")
display(
    test_result_df[
        test_result_df["exact_match"] == True
    ][
        [
            "unit_text_step2",
            "ground_truth_decoded",
            "prediction_decoded"
        ]
    ].sample(
        min(15, int((test_result_df["exact_match"] == True).sum())),
        random_state=42
    )
)

print("\n===== MẪU DỰ ĐOÁN SAI =====")
display(
    test_result_df[
        test_result_df["exact_match"] == False
    ][
        [
            "unit_text_step2",
            "ground_truth_decoded",
            "prediction_decoded",
            "num_wrong_aspects"
        ]
    ].sort_values(
        "num_wrong_aspects",
        ascending=False
    ).head(30)
)

In [ ]:
# ============================================================
# CELL 53: LƯU KẾT QUẢ ĐÁNH GIÁ TEST SET
# ============================================================

EVALUATION_DIR = "/content/drive/MyDrive/DACS/phobert_absa_final_loss_balanced/test_evaluation"
os.makedirs(EVALUATION_DIR, exist_ok=True)

# 1. Lưu bảng metric theo aspect
ASPECT_METRICS_PATH = os.path.join(
    EVALUATION_DIR,
    "test_aspect_summary_metrics.csv"
)
aspect_summary_df.to_csv(
    ASPECT_METRICS_PATH,
    index=False,
    encoding="utf-8-sig"
)

# 2. Lưu classification report từng aspect
for aspect, report_df in classification_reports.items():
    report_path = os.path.join(
        EVALUATION_DIR,
        f"classification_report_{aspect}.csv"
    )
    report_df.to_csv(
        report_path,
        encoding="utf-8-sig"
    )

# 3. Lưu bảng dự đoán chi tiết
TEST_PREDICTIONS_PATH = os.path.join(
    EVALUATION_DIR,
    "test_predictions_detail.csv"
)
test_result_df.to_csv(
    TEST_PREDICTIONS_PATH,
    index=False,
    encoding="utf-8-sig"
)

# 4. Lưu metric tổng quát
overall_metrics = {
    "mean_aspect_accuracy": float(mean_aspect_accuracy),
    "mean_aspect_macro_f1": float(mean_aspect_macro_f1),
    "mean_aspect_weighted_f1": float(mean_aspect_weighted_f1),
    "exact_match": float(exact_match)
}

OVERALL_METRICS_PATH = os.path.join(
    EVALUATION_DIR,
    "test_overall_metrics.json"
)

with open(OVERALL_METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(
        overall_metrics,
        f,
        ensure_ascii=False,
        indent=2
    )

print("✅ Đã lưu kết quả đánh giá:")
print(ASPECT_METRICS_PATH)
print(TEST_PREDICTIONS_PATH)
print(OVERALL_METRICS_PATH)

# Tokenize dataset cho ViSoBERT

In [ ]:
# ============================================================
# CELL 54: LOAD TRAIN/VAL/TEST CHO VISOBERT + TẠO LABEL VECTOR
# ============================================================

import os
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Load dữ liệu đã split
# ------------------------------------------------------------
SPLIT_DIR = "/content/drive/MyDrive/DACS/absa_splits"

TRAIN_PATH = os.path.join(SPLIT_DIR, "train.csv")
VAL_PATH = os.path.join(SPLIT_DIR, "validation.csv")
TEST_PATH = os.path.join(SPLIT_DIR, "test.csv")

for path in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Không tìm thấy file: {path}")

viso_train_df = pd.read_csv(TRAIN_PATH)
viso_val_df = pd.read_csv(VAL_PATH)
viso_test_df = pd.read_csv(TEST_PATH)

# ------------------------------------------------------------
# 2. Khai báo schema label
# ------------------------------------------------------------
ASPECTS = [
    "HOTEL",
    "LOCATION",
    "ROOMS",
    "FACILITIES",
    "FOOD&DRINKS",
    "SERVICE"
]

LABEL_ID_COLUMNS = [f"{aspect}_id" for aspect in ASPECTS]

# ------------------------------------------------------------
# 3. Tạo vector labels = [6 aspect ids]
# ------------------------------------------------------------
for split_name, dataframe in [
    ("train", viso_train_df),
    ("validation", viso_val_df),
    ("test", viso_test_df)
]:
    required_cols = {"text_model_input"} | set(LABEL_ID_COLUMNS)
    missing_cols = required_cols - set(dataframe.columns)

    if missing_cols:
        raise ValueError(f"{split_name} thiếu cột: {missing_cols}")

    dataframe[LABEL_ID_COLUMNS] = dataframe[LABEL_ID_COLUMNS].astype(int)
    dataframe["labels"] = dataframe[LABEL_ID_COLUMNS].values.tolist()

print("✅ Load dữ liệu và tạo labels cho ViSoBERT thành công")
print("Train:", viso_train_df.shape)
print("Validation:", viso_val_df.shape)
print("Test:", viso_test_df.shape)

display(
    viso_train_df[
        ["text_model_input", "FINAL_REVIEWED_SENTIMENTS", "labels"]
    ].head(10)
)

In [ ]:
# ============================================================
# CELL 55: LOAD VISOBERT TOKENIZER + ĐO TOKEN LENGTH + CHỌN MAX LENGTH
# ============================================================

!pip install -q transformers datasets

from transformers import AutoTokenizer
from tqdm.auto import tqdm

VISOBERT_MODEL_NAME = "uitnlp/visobert"

visobert_tokenizer = AutoTokenizer.from_pretrained(
    VISOBERT_MODEL_NAME
)

print("✅ Load ViSoBERT tokenizer thành công")
print("Tokenizer class:", visobert_tokenizer.__class__.__name__)
print("Vocab size:", visobert_tokenizer.vocab_size)

# ------------------------------------------------------------
# 1. Gom toàn bộ text train/val/test
# ------------------------------------------------------------
viso_all_texts = pd.concat([
    viso_train_df["text_model_input"],
    viso_val_df["text_model_input"],
    viso_test_df["text_model_input"]
], ignore_index=True).fillna("").astype(str).tolist()

# ------------------------------------------------------------
# 2. Đo độ dài token
# ------------------------------------------------------------
viso_token_lengths = []

for text in tqdm(viso_all_texts, desc="Đang đo token length cho ViSoBERT"):
    encoded = visobert_tokenizer(
        text,
        add_special_tokens=True,
        truncation=False
    )
    viso_token_lengths.append(len(encoded["input_ids"]))

viso_length_series = pd.Series(viso_token_lengths)

print("\nThống kê token length ViSoBERT:")
display(
    viso_length_series.describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    ).to_frame("token_length")
)

# ------------------------------------------------------------
# 3. Đề xuất MAX_LENGTH
# ------------------------------------------------------------
viso_p99 = int(viso_length_series.quantile(0.99))

if viso_p99 <= 64:
    VISOBERT_MAX_LENGTH = 64
elif viso_p99 <= 128:
    VISOBERT_MAX_LENGTH = 128
else:
    VISOBERT_MAX_LENGTH = 256

print("\nĐộ dài vượt ngưỡng:")
print("Số mẫu > 64 :", int((viso_length_series > 64).sum()))
print("Số mẫu > 128:", int((viso_length_series > 128).sum()))
print("Số mẫu > 256:", int((viso_length_series > 256).sum()))

print("\n✅ VISOBERT_MAX_LENGTH được đề xuất:", VISOBERT_MAX_LENGTH)

In [ ]:
# ============================================================
# CELL 56: TẠO DATASET VISOBERT + TOKENIZE TOÀN BỘ
# ============================================================

from datasets import Dataset, DatasetDict

# ------------------------------------------------------------
# 1. Chỉ giữ cột input và labels
# ------------------------------------------------------------
VISO_KEEP_COLUMNS = [
    "text_model_input",
    "labels"
]

visobert_dataset_dict = DatasetDict({
    "train": Dataset.from_pandas(
        viso_train_df[VISO_KEEP_COLUMNS].reset_index(drop=True)
    ),
    "validation": Dataset.from_pandas(
        viso_val_df[VISO_KEEP_COLUMNS].reset_index(drop=True)
    ),
    "test": Dataset.from_pandas(
        viso_test_df[VISO_KEEP_COLUMNS].reset_index(drop=True)
    )
})

print("Dataset trước tokenize:")
print(visobert_dataset_dict)

# ------------------------------------------------------------
# 2. Hàm tokenize riêng cho ViSoBERT
# ------------------------------------------------------------
def visobert_tokenize_batch(batch):
    return visobert_tokenizer(
        batch["text_model_input"],
        padding="max_length",
        truncation=True,
        max_length=VISOBERT_MAX_LENGTH
    )

# ------------------------------------------------------------
# 3. Tokenize toàn bộ dataset
# ------------------------------------------------------------
tokenized_visobert_datasets = visobert_dataset_dict.map(
    visobert_tokenize_batch,
    batched=True,
    desc="Đang tokenize dữ liệu cho ViSoBERT"
)

print("\n✅ Tokenize ViSoBERT hoàn tất")
print(tokenized_visobert_datasets)

print("\nVí dụ 1 mẫu sau tokenize:")
print(tokenized_visobert_datasets["train"][0])

In [ ]:
# ============================================================
# CELL 57: SET FORMAT PYTORCH + KIỂM TRA + LƯU TOKENIZED VISOBERT DATASET
# ============================================================

import os

# ------------------------------------------------------------
# 1. Bỏ text raw khỏi dataset tokenized
# ------------------------------------------------------------
tokenized_visobert_datasets = tokenized_visobert_datasets.remove_columns([
    "text_model_input"
])

# ------------------------------------------------------------
# 2. Chuyển sang PyTorch tensor
# ------------------------------------------------------------
tokenized_visobert_datasets.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)

# ------------------------------------------------------------
# 3. Kiểm tra 1 sample
# ------------------------------------------------------------
viso_sample = tokenized_visobert_datasets["train"][0]

print("✅ Dataset ViSoBERT đã sẵn sàng cho PyTorch")
print(tokenized_visobert_datasets)

print("\nKiểm tra 1 sample:")
print("input_ids shape:", viso_sample["input_ids"].shape)
print("attention_mask shape:", viso_sample["attention_mask"].shape)
print("labels:", viso_sample["labels"])
print("labels shape:", viso_sample["labels"].shape)

# ------------------------------------------------------------
# 4. Kiểm tra số mẫu có thể bị truncate
# ------------------------------------------------------------
viso_num_truncated_like = int((viso_length_series > VISOBERT_MAX_LENGTH).sum())
viso_ratio_truncated_like = viso_num_truncated_like / len(viso_length_series)

print("\nKiểm tra truncate:")
print("VISOBERT_MAX_LENGTH:", VISOBERT_MAX_LENGTH)
print("Số mẫu dài hơn VISOBERT_MAX_LENGTH:", viso_num_truncated_like)
print("Tỷ lệ có thể bị truncate:", round(viso_ratio_truncated_like * 100, 4), "%")

# ------------------------------------------------------------
# 5. Lưu checkpoint riêng cho ViSoBERT
# ------------------------------------------------------------
VISOBERT_TOKENIZED_SAVE_DIR = (
    "/content/drive/MyDrive/DACS/absa_tokenized_visobert_datasets"
)

tokenized_visobert_datasets.save_to_disk(
    VISOBERT_TOKENIZED_SAVE_DIR
)

print("\n✅ Đã lưu tokenized ViSoBERT dataset:")
print(VISOBERT_TOKENIZED_SAVE_DIR)

# Huấn luyện ViSoBERT cho bài toán ABSA

In [ ]:
# @title
# ============================================================
# CELL 58: CẤU HÌNH CHUNG CHO VISOBERT + KIỂM TRA GPU
# ============================================================

import os
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from transformers import AutoModel

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    print("✅ GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ Đang chạy bằng CPU. Train ViSoBERT sẽ rất chậm.")

# ------------------------------------------------------------
# 1. Cấu hình bài toán
# ------------------------------------------------------------
ASPECTS = [
    "HOTEL",
    "LOCATION",
    "ROOMS",
    "FACILITIES",
    "FOOD&DRINKS",
    "SERVICE"
]

ID2LABEL = {
    0: "NONE",
    1: "POSITIVE",
    2: "NEGATIVE",
    3: "NEUTRAL",
    4: "CONFLICT"
}

NUM_ASPECTS = len(ASPECTS)
NUM_CLASSES = len(ID2LABEL)

VISOBERT_MODEL_NAME = "uitnlp/visobert"

# Đã xác nhận ở Tác vụ 8
VISOBERT_MAX_LENGTH = 128

print("NUM_ASPECTS:", NUM_ASPECTS)
print("NUM_CLASSES:", NUM_CLASSES)
print("VISOBERT_MODEL_NAME:", VISOBERT_MODEL_NAME)
print("VISOBERT_MAX_LENGTH:", VISOBERT_MAX_LENGTH)

print("\nDataset ViSoBERT:")
print(tokenized_visobert_datasets)

In [ ]:

# ============================================================
# CELL 59: CLASS WEIGHTS + MODEL VISOBERT ABSA + METRICS
# ============================================================

from sklearn.metrics import f1_score, accuracy_score

# ------------------------------------------------------------
# 1. Lấy label train để tính class weights
# ------------------------------------------------------------
viso_train_labels_np = np.array(
    tokenized_visobert_datasets["train"]["labels"]
)

def visobert_build_class_weights(max_class_weight: float = 4.0) -> torch.Tensor:
    """
    Tính class weights riêng cho từng aspect.
    Output shape: [NUM_ASPECTS, NUM_CLASSES]
    """
    all_weights = []

    for aspect_idx in range(NUM_ASPECTS):
        y = viso_train_labels_np[:, aspect_idx]

        counts = np.bincount(y, minlength=NUM_CLASSES)
        total = counts.sum()

        weights = total / (NUM_CLASSES * np.maximum(counts, 1))
        weights = np.clip(weights, 0.0, max_class_weight)

        # Chuẩn hóa để trung bình weight xấp xỉ 1
        weights = weights / weights.mean()

        all_weights.append(weights)

    return torch.tensor(
        np.array(all_weights),
        dtype=torch.float32
    )

# ------------------------------------------------------------
# 2. Model ViSoBERT cho ABSA
# ------------------------------------------------------------
class ViSoBERTMultiAspectClassifier(nn.Module):
    def __init__(
        self,
        pretrained_model_name: str,
        num_aspects: int,
        num_classes: int,
        class_weights: torch.Tensor,
        dropout_prob: float = 0.3
    ):
        super().__init__()

        self.num_aspects = num_aspects
        self.num_classes = num_classes

        self.encoder = AutoModel.from_pretrained(
            pretrained_model_name
        )

        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(dropout_prob)

        self.classifier = nn.Linear(
            hidden_size,
            num_aspects * num_classes
        )

        self.register_buffer(
            "class_weights",
            class_weights
        )

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        labels=None
    ):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Token đầu câu, tương tự CLS representation
        cls_output = outputs.last_hidden_state[:, 0, :]

        cls_output = self.dropout(cls_output)

        logits = self.classifier(cls_output)
        logits = logits.view(
            -1,
            self.num_aspects,
            self.num_classes
        )

        loss = None

        if labels is not None:
            labels = labels.long()
            aspect_losses = []

            for aspect_idx in range(self.num_aspects):
                loss_fn = nn.CrossEntropyLoss(
                    weight=self.class_weights[aspect_idx]
                )

                aspect_loss = loss_fn(
                    logits[:, aspect_idx, :],
                    labels[:, aspect_idx]
                )

                aspect_losses.append(aspect_loss)

            loss = torch.stack(aspect_losses).mean()

        return {
            "loss": loss,
            "logits": logits
        }

# ------------------------------------------------------------
# 3. Metrics dùng chung với PhoBERT
# ------------------------------------------------------------
def visobert_compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    metrics = {}

    aspect_macro_f1_scores = []
    aspect_accuracy_scores = []

    for aspect_idx, aspect in enumerate(ASPECTS):
        y_true = labels[:, aspect_idx]
        y_pred = preds[:, aspect_idx]

        macro_f1 = f1_score(
            y_true,
            y_pred,
            labels=list(range(NUM_CLASSES)),
            average="macro",
            zero_division=0
        )

        accuracy = accuracy_score(
            y_true,
            y_pred
        )

        aspect_macro_f1_scores.append(macro_f1)
        aspect_accuracy_scores.append(accuracy)

        metrics[f"{aspect.lower()}_macro_f1"] = float(macro_f1)
        metrics[f"{aspect.lower()}_accuracy"] = float(accuracy)

    metrics["mean_aspect_macro_f1"] = float(
        np.mean(aspect_macro_f1_scores)
    )

    metrics["mean_aspect_accuracy"] = float(
        np.mean(aspect_accuracy_scores)
    )

    metrics["exact_match"] = float(
        np.mean(np.all(preds == labels, axis=1))
    )

    return metrics

print("✅ Đã khai báo:")
print("- visobert_build_class_weights")
print("- ViSoBERTMultiAspectClassifier")
print("- visobert_compute_metrics")

In [ ]:

# ============================================================
# CELL 60: HPO VISOBERT — LOSS-BALANCED
# ============================================================

!pip install -q optuna

from transformers import (
    TrainingArguments,
    Trainer,
    default_data_collator
)
import inspect

VISOBERT_HPO_OUTPUT_DIR = (
    "/content/drive/MyDrive/DACS/visobert_absa_hpo_loss_balanced"
)

# ------------------------------------------------------------
# 1. model_init cho từng HPO trial
# ------------------------------------------------------------
def visobert_model_init(trial=None):
    if trial is None:
        dropout_prob = 0.3
        max_class_weight = 4.0
    else:
        dropout_prob = trial.suggest_categorical(
            "dropout_prob",
            [0.2, 0.3, 0.4]
        )

        max_class_weight = trial.suggest_categorical(
            "max_class_weight",
            [3.0, 4.0, 5.0]
        )

    trial_class_weights = visobert_build_class_weights(
        max_class_weight=max_class_weight
    )

    return ViSoBERTMultiAspectClassifier(
        pretrained_model_name=VISOBERT_MODEL_NAME,
        num_aspects=NUM_ASPECTS,
        num_classes=NUM_CLASSES,
        class_weights=trial_class_weights,
        dropout_prob=dropout_prob
    )

# ------------------------------------------------------------
# 2. TrainingArguments nền cho HPO
# ------------------------------------------------------------
hpo_args_kwargs = dict(
    output_dir=VISOBERT_HPO_OUTPUT_DIR,

    learning_rate=1e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=5,

    weight_decay=0.05,
    warmup_ratio=0.1,

    logging_strategy="epoch",
    save_strategy="no",

    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED
)

training_args_signature = inspect.signature(
    TrainingArguments.__init__
).parameters

if "eval_strategy" in training_args_signature:
    hpo_args_kwargs["eval_strategy"] = "epoch"
else:
    hpo_args_kwargs["evaluation_strategy"] = "epoch"

visobert_hpo_training_args = TrainingArguments(
    **hpo_args_kwargs
)

# ------------------------------------------------------------
# 3. Trainer HPO
# ------------------------------------------------------------
visobert_hpo_trainer = Trainer(
    model_init=visobert_model_init,
    args=visobert_hpo_training_args,
    train_dataset=tokenized_visobert_datasets["train"],
    eval_dataset=tokenized_visobert_datasets["validation"],
    data_collator=default_data_collator,
    compute_metrics=visobert_compute_metrics
)

# ------------------------------------------------------------
# 4. Search space
# ------------------------------------------------------------
def visobert_hp_space(trial):
    return {
        "learning_rate": trial.suggest_float(
            "learning_rate",
            5e-6,
            2e-5,
            log=True
        ),

        "per_device_train_batch_size": trial.suggest_categorical(
            "per_device_train_batch_size",
            [8, 16]
        ),

        "weight_decay": trial.suggest_categorical(
            "weight_decay",
            [0.01, 0.05, 0.1]
        ),

        "warmup_ratio": trial.suggest_categorical(
            "warmup_ratio",
            [0.05, 0.1]
        )
    }

# ------------------------------------------------------------
# 5. Objective loss-balanced
# ------------------------------------------------------------
VISOBERT_ALPHA_LOSS = 0.25

def visobert_compute_objective(metrics):
    macro_f1 = metrics["eval_mean_aspect_macro_f1"]
    val_loss = metrics["eval_loss"]

    return macro_f1 - VISOBERT_ALPHA_LOSS * val_loss

# ------------------------------------------------------------
# 6. Số trial
# ------------------------------------------------------------
# Nếu cần tiết kiệm thời gian: để 6
# Nếu GPU ổn và muốn kỹ hơn: đổi thành 8
VISOBERT_N_TRIALS = 6

visobert_best_run = visobert_hpo_trainer.hyperparameter_search(
    direction="maximize",
    backend="optuna",
    hp_space=visobert_hp_space,
    compute_objective=visobert_compute_objective,
    n_trials=VISOBERT_N_TRIALS
)

print("✅ HPO ViSoBERT hoàn tất")

print("\nBest composite objective:")
print(visobert_best_run.objective)

print("\nBest hyperparameters:")
print(visobert_best_run.hyperparameters)

In [ ]:

# ============================================================
# CELL 61: TRAIN FINAL VISOBERT BẰNG BEST HYPERPARAMETERS
# ============================================================

from transformers import EarlyStoppingCallback

visobert_best_params = visobert_best_run.hyperparameters

VISO_BEST_LEARNING_RATE = visobert_best_params["learning_rate"]
VISO_BEST_BATCH_SIZE = visobert_best_params["per_device_train_batch_size"]
VISO_BEST_WEIGHT_DECAY = visobert_best_params["weight_decay"]
VISO_BEST_WARMUP_RATIO = visobert_best_params["warmup_ratio"]
VISO_BEST_DROPOUT = visobert_best_params["dropout_prob"]
VISO_BEST_MAX_CLASS_WEIGHT = visobert_best_params["max_class_weight"]

print("Best params dùng cho final ViSoBERT:")
print(json.dumps(visobert_best_params, indent=2))

# ------------------------------------------------------------
# 1. Tạo class weights final
# ------------------------------------------------------------
visobert_best_class_weights = visobert_build_class_weights(
    max_class_weight=VISO_BEST_MAX_CLASS_WEIGHT
)

visobert_class_weight_df = pd.DataFrame(
    visobert_best_class_weights.numpy(),
    index=ASPECTS,
    columns=[ID2LABEL[i] for i in range(NUM_CLASSES)]
)

print("\nClass weights final ViSoBERT:")
display(visobert_class_weight_df)

# ------------------------------------------------------------
# 2. Khởi tạo final model mới
# ------------------------------------------------------------
visobert_final_model = ViSoBERTMultiAspectClassifier(
    pretrained_model_name=VISOBERT_MODEL_NAME,
    num_aspects=NUM_ASPECTS,
    num_classes=NUM_CLASSES,
    class_weights=visobert_best_class_weights,
    dropout_prob=VISO_BEST_DROPOUT
)

# ------------------------------------------------------------
# 3. TrainingArguments final
# ------------------------------------------------------------
VISOBERT_FINAL_OUTPUT_DIR = (
    "/content/drive/MyDrive/DACS/visobert_absa_final_loss_balanced"
)

final_args_kwargs = dict(
    output_dir=VISOBERT_FINAL_OUTPUT_DIR,

    learning_rate=VISO_BEST_LEARNING_RATE,
    per_device_train_batch_size=VISO_BEST_BATCH_SIZE,
    per_device_eval_batch_size=16,
    num_train_epochs=10,

    weight_decay=VISO_BEST_WEIGHT_DECAY,
    warmup_ratio=VISO_BEST_WARMUP_RATIO,

    logging_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,

    save_total_limit=2,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED
)

if "eval_strategy" in training_args_signature:
    final_args_kwargs["eval_strategy"] = "epoch"
else:
    final_args_kwargs["evaluation_strategy"] = "epoch"

visobert_final_training_args = TrainingArguments(
    **final_args_kwargs
)

# ------------------------------------------------------------
# 4. Trainer final
# ------------------------------------------------------------
visobert_final_trainer = Trainer(
    model=visobert_final_model,
    args=visobert_final_training_args,
    train_dataset=tokenized_visobert_datasets["train"],
    eval_dataset=tokenized_visobert_datasets["validation"],
    data_collator=default_data_collator,
    compute_metrics=visobert_compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ]
)

# ------------------------------------------------------------
# 5. Train
# ------------------------------------------------------------
visobert_final_train_result = visobert_final_trainer.train()

print("\n✅ Final training ViSoBERT hoàn tất")

# ------------------------------------------------------------
# 6. Evaluate validation và test
# ------------------------------------------------------------
visobert_validation_metrics = visobert_final_trainer.evaluate(
    tokenized_visobert_datasets["validation"],
    metric_key_prefix="validation"
)

visobert_test_metrics = visobert_final_trainer.evaluate(
    tokenized_visobert_datasets["test"],
    metric_key_prefix="test"
)

print("\nValidation metrics — ViSoBERT:")
for k, v in visobert_validation_metrics.items():
    print(f"{k}: {v}")

print("\nTest metrics — ViSoBERT:")
for k, v in visobert_test_metrics.items():
    print(f"{k}: {v}")

In [ ]:
# ============================================================
# KIỂM TRA LOAD BEST MODEL AT END CÓ HOẠT ĐỘNG KHÔNG
# ============================================================

print("load_best_model_at_end:",
      visobert_final_trainer.args.load_best_model_at_end)

print("metric_for_best_model:",
      visobert_final_trainer.args.metric_for_best_model)

print("greater_is_better:",
      visobert_final_trainer.args.greater_is_better)

print("\nBest model checkpoint:")
print(visobert_final_trainer.state.best_model_checkpoint)

print("\nBest metric:")
print(visobert_final_trainer.state.best_metric)

In [ ]:
# ============================================================
# CELL 62: VẼ LOSS CURVE VÀ MACRO F1 CURVE CHO VISOBERT
# ============================================================

import matplotlib.pyplot as plt

viso_log_history = visobert_final_trainer.state.log_history

viso_train_epochs = []
viso_train_losses = []

viso_val_epochs = []
viso_val_losses = []

viso_val_f1_epochs = []
viso_val_macro_f1s = []

for log in viso_log_history:
    if "loss" in log and "epoch" in log:
        viso_train_epochs.append(log["epoch"])
        viso_train_losses.append(log["loss"])

    if "eval_loss" in log and "epoch" in log:
        viso_val_epochs.append(log["epoch"])
        viso_val_losses.append(log["eval_loss"])

    if "eval_mean_aspect_macro_f1" in log and "epoch" in log:
        viso_val_f1_epochs.append(log["epoch"])
        viso_val_macro_f1s.append(
            log["eval_mean_aspect_macro_f1"]
        )

# ------------------------------------------------------------
# 1. Train Loss vs Validation Loss
# ------------------------------------------------------------
plt.figure(figsize=(8, 5))
plt.plot(
    viso_train_epochs,
    viso_train_losses,
    marker="o",
    label="Train Loss"
)
plt.plot(
    viso_val_epochs,
    viso_val_losses,
    marker="o",
    label="Validation Loss"
)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("ViSoBERT — Train Loss vs Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

# ------------------------------------------------------------
# 2. Validation Macro F1
# ------------------------------------------------------------
plt.figure(figsize=(8, 5))
plt.plot(
    viso_val_f1_epochs,
    viso_val_macro_f1s,
    marker="o",
    label="Validation Mean Aspect Macro F1"
)
plt.xlabel("Epoch")
plt.ylabel("Macro F1")
plt.title("ViSoBERT — Validation Mean Aspect Macro F1")
plt.legend()
plt.grid(True)
plt.show()

# ------------------------------------------------------------
# 3. Bảng theo epoch
# ------------------------------------------------------------
visobert_training_curve_df = pd.DataFrame({
    "epoch": viso_val_epochs,
    "validation_loss": viso_val_losses,
    "validation_mean_macro_f1": viso_val_macro_f1s
})

display(visobert_training_curve_df)

In [ ]:
# ============================================================
# CELL 63: LƯU ARTIFACTS VISOBERT
# ============================================================

VISOBERT_FINAL_SAVE_DIR = (
    "/content/drive/MyDrive/DACS/"
    "visobert_absa_final_loss_balanced/final_artifacts"
)

os.makedirs(VISOBERT_FINAL_SAVE_DIR, exist_ok=True)

# ------------------------------------------------------------
# 1. Lưu model state
# ------------------------------------------------------------
VISOBERT_MODEL_PATH = os.path.join(
    VISOBERT_FINAL_SAVE_DIR,
    "visobert_absa_multiaspect_loss_balanced.pt"
)

torch.save(
    {
        "model_state_dict": visobert_final_trainer.model.state_dict(),
        "model_name": VISOBERT_MODEL_NAME,
        "num_aspects": NUM_ASPECTS,
        "num_classes": NUM_CLASSES,
        "aspects": ASPECTS,
        "id2label": ID2LABEL,
        "max_length": VISOBERT_MAX_LENGTH,
        "best_hyperparameters": visobert_best_params,
        "class_weights": visobert_best_class_weights.cpu().numpy().tolist(),
        "selection_policy": {
            "hpo_objective": "macro_f1 - 0.25 * eval_loss",
            "final_checkpoint_metric": "eval_loss"
        }
    },
    VISOBERT_MODEL_PATH
)

# ------------------------------------------------------------
# 2. Lưu best hyperparameters
# ------------------------------------------------------------
VISOBERT_BEST_PARAMS_PATH = os.path.join(
    VISOBERT_FINAL_SAVE_DIR,
    "visobert_best_hyperparameters_loss_balanced.json"
)

with open(
    VISOBERT_BEST_PARAMS_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        visobert_best_params,
        f,
        ensure_ascii=False,
        indent=2
    )

# ------------------------------------------------------------
# 3. Lưu metrics
# ------------------------------------------------------------
VISOBERT_METRICS_PATH = os.path.join(
    VISOBERT_FINAL_SAVE_DIR,
    "visobert_final_metrics_loss_balanced.json"
)

with open(
    VISOBERT_METRICS_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        {
            "validation_metrics": visobert_validation_metrics,
            "test_metrics": visobert_test_metrics
        },
        f,
        ensure_ascii=False,
        indent=2
    )

# ------------------------------------------------------------
# 4. Lưu trainer log history
# ------------------------------------------------------------
VISOBERT_LOG_HISTORY_PATH = os.path.join(
    VISOBERT_FINAL_SAVE_DIR,
    "visobert_trainer_log_history_loss_balanced.json"
)

with open(
    VISOBERT_LOG_HISTORY_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        visobert_final_trainer.state.log_history,
        f,
        ensure_ascii=False,
        indent=2
    )

print("✅ Đã lưu artifacts ViSoBERT:")
print(VISOBERT_MODEL_PATH)
print(VISOBERT_BEST_PARAMS_PATH)
print(VISOBERT_METRICS_PATH)
print(VISOBERT_LOG_HISTORY_PATH)

# Đánh giá ViSoBERT trên tập test

In [ ]:
# ============================================================
# CELL 64: PREDICT VISOBERT TRÊN TEST SET
# ============================================================

import numpy as np
import pandas as pd

# 1. Chạy dự đoán
visobert_test_prediction_output = visobert_final_trainer.predict(
    tokenized_visobert_datasets["test"]
)

# 2. Lấy logits và labels thật
visobert_test_logits = visobert_test_prediction_output.predictions
visobert_test_labels = visobert_test_prediction_output.label_ids

# 3. Logits -> label id dự đoán
visobert_test_preds = np.argmax(
    visobert_test_logits,
    axis=-1
)

print("✅ Predict test set cho ViSoBERT hoàn tất")
print("visobert_test_logits shape:", visobert_test_logits.shape)
print("visobert_test_labels shape:", visobert_test_labels.shape)
print("visobert_test_preds shape:", visobert_test_preds.shape)

In [ ]:
# ============================================================
# CELL 65: TEST METRICS TỔNG QUÁT + THEO ASPECT CHO VISOBERT
# ============================================================

from sklearn.metrics import f1_score, accuracy_score

ASPECTS = [
    "HOTEL",
    "LOCATION",
    "ROOMS",
    "FACILITIES",
    "FOOD&DRINKS",
    "SERVICE"
]

ID2LABEL = {
    0: "NONE",
    1: "POSITIVE",
    2: "NEGATIVE",
    3: "NEUTRAL",
    4: "CONFLICT"
}

NUM_CLASSES = len(ID2LABEL)

visobert_aspect_summary_rows = []

for aspect_idx, aspect in enumerate(ASPECTS):
    y_true = visobert_test_labels[:, aspect_idx]
    y_pred = visobert_test_preds[:, aspect_idx]

    macro_f1 = f1_score(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        average="weighted",
        zero_division=0
    )

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    visobert_aspect_summary_rows.append({
        "aspect": aspect,
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1
    })

visobert_aspect_summary_df = pd.DataFrame(
    visobert_aspect_summary_rows
)

visobert_mean_aspect_macro_f1 = (
    visobert_aspect_summary_df["macro_f1"].mean()
)

visobert_mean_aspect_accuracy = (
    visobert_aspect_summary_df["accuracy"].mean()
)

visobert_mean_aspect_weighted_f1 = (
    visobert_aspect_summary_df["weighted_f1"].mean()
)

visobert_exact_match = np.mean(
    np.all(
        visobert_test_preds == visobert_test_labels,
        axis=1
    )
)

print("===== VISOBERT — TEST SET OVERALL METRICS =====")
print(f"Mean Aspect Accuracy   : {visobert_mean_aspect_accuracy:.4f}")
print(f"Mean Aspect Macro F1   : {visobert_mean_aspect_macro_f1:.4f}")
print(f"Mean Aspect Weighted F1: {visobert_mean_aspect_weighted_f1:.4f}")
print(f"Exact Match            : {visobert_exact_match:.4f}")

print("\n===== VISOBERT — TEST METRICS THEO TỪNG ASPECT =====")
display(
    visobert_aspect_summary_df
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
)

In [ ]:
# ============================================================
# CELL 66: CLASSIFICATION REPORT TỪNG ASPECT — VISOBERT
# ============================================================

from sklearn.metrics import classification_report

visobert_classification_reports = {}

LABEL_NAMES = [ID2LABEL[i] for i in range(NUM_CLASSES)]

for aspect_idx, aspect in enumerate(ASPECTS):
    y_true = visobert_test_labels[:, aspect_idx]
    y_pred = visobert_test_preds[:, aspect_idx]

    report_dict = classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=LABEL_NAMES,
        output_dict=True,
        zero_division=0
    )

    report_df = pd.DataFrame(report_dict).transpose()

    visobert_classification_reports[aspect] = report_df

    print("=" * 100)
    print(f"VISOBERT — CLASSIFICATION REPORT — {aspect}")
    display(report_df.round(4))

In [ ]:
# ============================================================
# CELL 67: CONFUSION MATRIX TỪNG ASPECT — VISOBERT
# ============================================================

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

for aspect_idx, aspect in enumerate(ASPECTS):
    y_true = visobert_test_labels[:, aspect_idx]
    y_pred = visobert_test_preds[:, aspect_idx]

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES))
    )

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=LABEL_NAMES
    )

    plt.figure(figsize=(7, 6))
    disp.plot(
        values_format="d",
        ax=plt.gca(),
        colorbar=False
    )
    plt.title(f"ViSoBERT — Confusion Matrix — {aspect}")
    plt.xticks(rotation=45)
    plt.grid(False)
    plt.show()

In [ ]:
# ============================================================
# CELL 68: TẠO BẢNG PREDICTION DETAIL + LƯU EVALUATION VISOBERT
# ============================================================

import os
import json

# ------------------------------------------------------------
# 1. Load lại test.csv để lấy text gốc
# ------------------------------------------------------------
TEST_CSV_PATH = "/content/drive/MyDrive/DACS/absa_splits/test.csv"

visobert_test_raw_df = pd.read_csv(
    TEST_CSV_PATH
).reset_index(drop=True)

if len(visobert_test_raw_df) != len(visobert_test_preds):
    raise ValueError(
        f"Số dòng test.csv ({len(visobert_test_raw_df)}) "
        f"khác số prediction ({len(visobert_test_preds)})"
    )

# ------------------------------------------------------------
# 2. Hàm decode vector label -> chuỗi dễ đọc
# ------------------------------------------------------------
def decode_label_row_for_visobert(label_row):
    pairs = []

    for aspect_idx, label_id in enumerate(label_row):
        label_name = ID2LABEL[int(label_id)]

        if label_name != "NONE":
            pairs.append(
                f"{ASPECTS[aspect_idx]}:{label_name}"
            )

    return "; ".join(pairs) if pairs else "NONE"

# ------------------------------------------------------------
# 3. Tạo bảng kết quả chi tiết
# ------------------------------------------------------------
visobert_test_result_df = visobert_test_raw_df[
    [
        "review_index",
        "unit_id",
        "unit_text_step2",
        "text_model_input",
        "FINAL_REVIEWED_SENTIMENTS"
    ]
].copy()

visobert_test_result_df["ground_truth_decoded"] = [
    decode_label_row_for_visobert(row)
    for row in visobert_test_labels
]

visobert_test_result_df["prediction_decoded"] = [
    decode_label_row_for_visobert(row)
    for row in visobert_test_preds
]

visobert_test_result_df["exact_match"] = np.all(
    visobert_test_preds == visobert_test_labels,
    axis=1
)

visobert_test_result_df["num_wrong_aspects"] = (
    visobert_test_preds != visobert_test_labels
).sum(axis=1)

print("✅ Đã tạo bảng prediction detail cho ViSoBERT")
print("Số dòng test:", len(visobert_test_result_df))
print("Exact match:", round(
    visobert_test_result_df["exact_match"].mean(),
    4
))

display(
    visobert_test_result_df.head(20)
)

# ------------------------------------------------------------
# 4. Xem nhanh các lỗi nặng nhất
# ------------------------------------------------------------
print("\n===== VISOBERT — MẪU DỰ ĐOÁN SAI NHIỀU NHẤT =====")
display(
    visobert_test_result_df[
        visobert_test_result_df["exact_match"] == False
    ][
        [
            "unit_text_step2",
            "ground_truth_decoded",
            "prediction_decoded",
            "num_wrong_aspects"
        ]
    ]
    .sort_values("num_wrong_aspects", ascending=False)
    .head(30)
)

# ------------------------------------------------------------
# 5. Lưu kết quả evaluation
# ------------------------------------------------------------
VISOBERT_EVAL_DIR = (
    "/content/drive/MyDrive/DACS/"
    "visobert_absa_final_loss_balanced/test_evaluation"
)

os.makedirs(VISOBERT_EVAL_DIR, exist_ok=True)

# Metric theo aspect
visobert_aspect_summary_df.to_csv(
    os.path.join(
        VISOBERT_EVAL_DIR,
        "visobert_test_aspect_summary_metrics.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

# Classification reports
for aspect, report_df in visobert_classification_reports.items():
    report_df.to_csv(
        os.path.join(
            VISOBERT_EVAL_DIR,
            f"visobert_classification_report_{aspect}.csv"
        ),
        encoding="utf-8-sig"
    )

# Prediction detail
visobert_test_result_df.to_csv(
    os.path.join(
        VISOBERT_EVAL_DIR,
        "visobert_test_predictions_detail.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

# Overall metrics
visobert_overall_metrics = {
    "mean_aspect_accuracy": float(
        visobert_mean_aspect_accuracy
    ),
    "mean_aspect_macro_f1": float(
        visobert_mean_aspect_macro_f1
    ),
    "mean_aspect_weighted_f1": float(
        visobert_mean_aspect_weighted_f1
    ),
    "exact_match": float(
        visobert_exact_match
    )
}

with open(
    os.path.join(
        VISOBERT_EVAL_DIR,
        "visobert_test_overall_metrics.json"
    ),
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        visobert_overall_metrics,
        f,
        ensure_ascii=False,
        indent=2
    )

print("\n✅ Đã lưu toàn bộ kết quả đánh giá ViSoBERT tại:")
print(VISOBERT_EVAL_DIR)

# Trực quan hóa và so sánh PhoBERT vs ViSoBERT

In [ ]:
# ============================================================
# CELL 68.5: LOAD LẠI KẾT QUẢ ĐÁNH GIÁ PHOBERT TỪ DRIVE
# ============================================================

import os
import json
import pandas as pd

# ------------------------------------------------------------
# 1. Khai báo thư mục evaluation PhoBERT đã lưu trước đó
# ------------------------------------------------------------
PHOBERT_EVAL_DIR = (
    "/content/drive/MyDrive/DACS/"
    "phobert_absa_final_loss_balanced/test_evaluation"
)

PHOBERT_OVERALL_METRICS_PATH = os.path.join(
    PHOBERT_EVAL_DIR,
    "test_overall_metrics.json"
)

PHOBERT_ASPECT_METRICS_PATH = os.path.join(
    PHOBERT_EVAL_DIR,
    "test_aspect_summary_metrics.csv"
)

PHOBERT_PREDICTIONS_PATH = os.path.join(
    PHOBERT_EVAL_DIR,
    "test_predictions_detail.csv"
)

# ------------------------------------------------------------
# 2. Kiểm tra file tồn tại
# ------------------------------------------------------------
required_paths = [
    PHOBERT_OVERALL_METRICS_PATH,
    PHOBERT_ASPECT_METRICS_PATH,
    PHOBERT_PREDICTIONS_PATH
]

for path in required_paths:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Không tìm thấy file PhoBERT: {path}")

# ------------------------------------------------------------
# 3. Load overall metrics PhoBERT
# ------------------------------------------------------------
with open(
    PHOBERT_OVERALL_METRICS_PATH,
    "r",
    encoding="utf-8"
) as f:
    phobert_overall_metrics = json.load(f)

# ------------------------------------------------------------
# 4. Load bảng metric theo aspect PhoBERT
#    Đặt đúng tên biến cũ để Cell 71 chạy lại được
# ------------------------------------------------------------
aspect_summary_df = pd.read_csv(
    PHOBERT_ASPECT_METRICS_PATH
)

# ------------------------------------------------------------
# 5. Load bảng prediction detail PhoBERT
#    Đặt đúng tên biến cũ để Cell 73 chạy lại được
# ------------------------------------------------------------
test_result_df = pd.read_csv(
    PHOBERT_PREDICTIONS_PATH
)

# ------------------------------------------------------------
# 6. Gán lại các biến metric chính của PhoBERT
# ------------------------------------------------------------
phobert_mean_aspect_accuracy = phobert_overall_metrics[
    "mean_aspect_accuracy"
]

phobert_mean_aspect_macro_f1 = phobert_overall_metrics[
    "mean_aspect_macro_f1"
]

phobert_mean_aspect_weighted_f1 = phobert_overall_metrics[
    "mean_aspect_weighted_f1"
]

phobert_exact_match = phobert_overall_metrics[
    "exact_match"
]

# ------------------------------------------------------------
# 7. Hiển thị để kiểm tra
# ------------------------------------------------------------
print("✅ Đã load lại kết quả PhoBERT từ Drive")

print("\nPhoBERT overall metrics:")
print(f"Mean Aspect Accuracy   : {phobert_mean_aspect_accuracy:.4f}")
print(f"Mean Aspect Macro F1   : {phobert_mean_aspect_macro_f1:.4f}")
print(f"Mean Aspect Weighted F1: {phobert_mean_aspect_weighted_f1:.4f}")
print(f"Exact Match            : {phobert_exact_match:.4f}")

print("\nPhoBERT aspect summary:")
display(aspect_summary_df)

print("\nPhoBERT prediction detail:")
display(test_result_df.head(10))

In [ ]:
# ============================================================
# CELL 69: BẢNG SO SÁNH METRIC TỔNG PHOBERT VS VISOBERT
# ============================================================

comparison_overall_df = pd.DataFrame([
    {
        "model": "PhoBERT",
        "mean_aspect_accuracy": phobert_mean_aspect_accuracy,
        "mean_aspect_macro_f1": phobert_mean_aspect_macro_f1,
        "mean_aspect_weighted_f1": phobert_mean_aspect_weighted_f1,
        "exact_match": phobert_exact_match
    },
    {
        "model": "ViSoBERT",
        "mean_aspect_accuracy": visobert_mean_aspect_accuracy,
        "mean_aspect_macro_f1": visobert_mean_aspect_macro_f1,
        "mean_aspect_weighted_f1": visobert_mean_aspect_weighted_f1,
        "exact_match": visobert_exact_match
    }
])

display(comparison_overall_df.round(4))

In [ ]:
# ============================================================
# CELL 70: BIỂU ĐỒ SO SÁNH METRIC TỔNG
# ============================================================

import matplotlib.pyplot as plt

metrics_to_plot = [
    "mean_aspect_accuracy",
    "mean_aspect_macro_f1",
    "mean_aspect_weighted_f1",
    "exact_match"
]

plot_df = comparison_overall_df.set_index("model")[metrics_to_plot].T

ax = plot_df.plot(
    kind="bar",
    figsize=(10, 6)
)

plt.title("PhoBERT vs ViSoBERT — Overall Test Metrics")
plt.ylabel("Score")
plt.xlabel("Metric")
plt.ylim(0, 1)
plt.xticks(rotation=30, ha="right")
plt.grid(axis="y")
plt.legend(title="Model")
plt.show()

In [ ]:
# ============================================================
# CELL 71: SO SÁNH MACRO F1 THEO TỪNG ASPECT
# ============================================================

phobert_aspect_compare_df = aspect_summary_df[
    ["aspect", "macro_f1"]
].rename(
    columns={"macro_f1": "PhoBERT_macro_f1"}
)

visobert_aspect_compare_df = visobert_aspect_summary_df[
    ["aspect", "macro_f1"]
].rename(
    columns={"macro_f1": "ViSoBERT_macro_f1"}
)

comparison_aspect_f1_df = phobert_aspect_compare_df.merge(
    visobert_aspect_compare_df,
    on="aspect",
    how="inner"
)

comparison_aspect_f1_df["difference_phobert_minus_visobert"] = (
    comparison_aspect_f1_df["PhoBERT_macro_f1"]
    - comparison_aspect_f1_df["ViSoBERT_macro_f1"]
)

display(
    comparison_aspect_f1_df
    .sort_values("difference_phobert_minus_visobert", ascending=False)
    .reset_index(drop=True)
    .round(4)
)

In [ ]:
# ============================================================
# CELL 72: BIỂU ĐỒ SO SÁNH MACRO F1 THEO ASPECT
# ============================================================

aspect_plot_df = comparison_aspect_f1_df.set_index("aspect")[
    ["PhoBERT_macro_f1", "ViSoBERT_macro_f1"]
]

ax = aspect_plot_df.plot(
    kind="bar",
    figsize=(11, 6)
)

plt.title("PhoBERT vs ViSoBERT — Macro F1 theo từng Aspect")
plt.ylabel("Macro F1")
plt.xlabel("Aspect")
plt.ylim(0, 1)
plt.xticks(rotation=30, ha="right")
plt.grid(axis="y")
plt.legend(title="Model")
plt.show()

In [ ]:
# ============================================================
# CELL 73: SO SÁNH CASE STUDY PREDICTION PHOBERT VS VISOBERT
# ============================================================

phobert_compare_cases = test_result_df[
    [
        "review_index",
        "unit_id",
        "unit_text_step2",
        "ground_truth_decoded",
        "prediction_decoded",
        "exact_match"
    ]
].rename(
    columns={
        "prediction_decoded": "phobert_prediction",
        "exact_match": "phobert_exact_match"
    }
)

visobert_compare_cases = visobert_test_result_df[
    [
        "review_index",
        "unit_id",
        "prediction_decoded",
        "exact_match"
    ]
].rename(
    columns={
        "prediction_decoded": "visobert_prediction",
        "exact_match": "visobert_exact_match"
    }
)

model_case_compare_df = phobert_compare_cases.merge(
    visobert_compare_cases,
    on=["review_index", "unit_id"],
    how="inner"
)

# ------------------------------------------------------------
# 1. PhoBERT đúng, ViSoBERT sai
# ------------------------------------------------------------
print("===== PhoBERT đúng, ViSoBERT sai =====")
display(
    model_case_compare_df[
        (model_case_compare_df["phobert_exact_match"] == True)
        & (model_case_compare_df["visobert_exact_match"] == False)
    ][
        [
            "unit_text_step2",
            "ground_truth_decoded",
            "phobert_prediction",
            "visobert_prediction"
        ]
    ].head(20)
)

# ------------------------------------------------------------
# 2. ViSoBERT đúng, PhoBERT sai
# ------------------------------------------------------------
print("\n===== ViSoBERT đúng, PhoBERT sai =====")
display(
    model_case_compare_df[
        (model_case_compare_df["phobert_exact_match"] == False)
        & (model_case_compare_df["visobert_exact_match"] == True)
    ][
        [
            "unit_text_step2",
            "ground_truth_decoded",
            "phobert_prediction",
            "visobert_prediction"
        ]
    ].head(20)
)

# ------------------------------------------------------------
# 3. Cả hai cùng sai
# ------------------------------------------------------------
print("\n===== Cả PhoBERT và ViSoBERT cùng sai =====")
display(
    model_case_compare_df[
        (model_case_compare_df["phobert_exact_match"] == False)
        & (model_case_compare_df["visobert_exact_match"] == False)
    ][
        [
            "unit_text_step2",
            "ground_truth_decoded",
            "phobert_prediction",
            "visobert_prediction"
        ]
    ].head(20)
)

In [ ]:
# ============================================================
# CELL 74: LƯU KẾT QUẢ SO SÁNH PHOBERT VS VISOBERT
# ============================================================

COMPARISON_DIR = "/content/drive/MyDrive/DACS/model_comparison"
os.makedirs(COMPARISON_DIR, exist_ok=True)

comparison_overall_df.to_csv(
    os.path.join(COMPARISON_DIR, "overall_metric_comparison.csv"),
    index=False,
    encoding="utf-8-sig"
)

comparison_aspect_f1_df.to_csv(
    os.path.join(COMPARISON_DIR, "aspect_macro_f1_comparison.csv"),
    index=False,
    encoding="utf-8-sig"
)

model_case_compare_df.to_csv(
    os.path.join(COMPARISON_DIR, "prediction_case_comparison.csv"),
    index=False,
    encoding="utf-8-sig"
)

print("✅ Đã lưu kết quả so sánh tại:")
print(COMPARISON_DIR)